# Chapter 1 · Standard FunnyBird CBM: controlled concept backwash

**Result in one sentence.** Controlled concept backwash exists in this
seed-1 Standard Koh Joint CBM, but it is graded rather than an all-parts-
behave-the-same effect: the inserted part moves the raw comparison
donorward, yet the old source remains higher in 50.2% of tail swaps,
20.0% of beak, 8.9% of eye, 3.2% of foot, and 1.9% of wing swaps.

**Required predicates and boundary.** The claim requires both
`response_delta>0` and final margin `m_cf<0` on the same validated
replacement. Visibility, label/mask conflict, and exact-value difficulty
align with the graded ordering; only visibility improves the declared
held-out margin predictor. These measurements do not make the residual
zero. Species information is abundant in raw concept scores,
but the current held-out categorical test gives source species no
generalizing explanatory credit. The report therefore concludes that
backwash exists, not that every cause is fully or causally identified.

**Why begin with a synthetic dataset?** FunnyBird is deliberately
contrived. Its renderer lets us change one named part while holding the
body, pose, camera, and background fixed. That makes it possible to
define and verify backwash more precisely than a natural photograph
permits. This chapter uses that privileged setting to establish the
event, calibrate warning signs, and motivate corrections; it does not
estimate natural-world prevalence.

**Starting question.** When one FunnyBird part is replaced while body,
pose, camera, and background stay fixed, what does the corresponding
concept output do? We do not label the result “backwash” unless the
predeclared response and final-margin conditions both hold.

**Population.** Standard non-RL CBM, seed 1. The discovery chain contains
no MCBM result and no visibility-aware relabelled model; MCBM appears only
in the final handoff to notebook 03. No seed-level uncertainty is available
yet, so reused swap rows are not presented as independent error bars.
“Standard” here means training with the original concept labels. “Non-RL”
means those labels were not changed according to part visibility.

**What this design can establish.** FunnyBird's renderer permits a
controlled donor-part replacement. A validated positive donor response
plus a remaining source preference can establish the event. Visibility,
label conflict, exact value, support, and species are investigated only
after the event is measured; most remain proposed contributors unless
independently manipulated.


## How this chapter fits the complete investigation

This report series deliberately moves from a setting with unusually complete
counterfactual information to settings where less can be known:

1. **FunnyBird Standard CBM (this chapter):** use the renderer to establish a
   precise controlled event and learn which observable warning signs accompany
   it.
2. **FunnyBird MCBM:** test whether compressing unnecessary information in the
   concept representation changes the same fixed-render outcome.
3. **FunnyBird RLv2 and relabelled MCBM:** test whether positive labels attached
   to invisible parts are a causal contributor and whether relabelling and
   minimality address different routes.
4. **CUB70:** carry only the FunnyBird-calibrated questions into natural bird
   photographs. Released masks permit visibility and context comparisons, but
   there is no same-image donor-part replacement.
5. **Full CUB:** test whether the CUB70 observations survive 200 species and
   weaker matched support, and report which mechanisms are no longer
   identifiable from the available data.

FunnyBird is therefore a **calibration laboratory**, not an estimate of how
often backwash occurs in ordinary photographs. CUB70 is the natural-image
bridge, and Full CUB is the robustness and identifiability test. As information
decreases, the claims narrow: FunnyBird can establish the controlled event;
CUB can only test explicitly labelled observational signatures.

### Correction hypotheses carried into later chapters

| FunnyBird warning sign | Proposed correction | Later test |
|---|---|---|
| positive concepts remain labelled when their part is invisible | visibility-aware RLv2 labels | matched Standard/RLv2 fixed swaps |
| raw-score magnitudes contain species information beyond the named labels | MCBM minimality/compression | gamma-dependent compression and fixed-swap response |
| both routes operate | combine relabelling and minimality | relabelled MCBM comparison |
| a residual remains after both | direct spatial or swap-consistency supervision may be needed | future method hypothesis, not a result of this chapter |

No later model is allowed to replace the Standard-CBM discovery below. Each
later chapter inherits a question from this one and must state which operation
its own dataset actually permits.


## The implemented standard CBM and the notation used below

This report uses the accepted **ResNet-50 Koh-architecture Joint CBM**, not the
CBM class from `minimal_cbm` and not an MCBM. For image `i`, the ResNet encoder
emits one raw logit for each of the 26 exact FunnyBird concepts. The single
linear species head reads those same 26 raw logits:

CBM means **concept bottleneck model**: instead of predicting species directly
from unspecified image features, it first produces named concept scores and
then predicts species from that bottleneck of scores. “Koh architecture” names
the published CBM design whose concept and class path is preserved here.

In ordinary language, ResNet-50 is the image-processing network. “Joint” means
the image-to-concept part and concept-to-species part are trained together
rather than in separate stages. A “linear species head” is one weighted sum per
species; it receives only the 26 concept scores, with no hidden nonlinear layer.

```text
image x_i
   |
   v
ResNet-50 image encoder
   |
   v
26 raw concept logits z_i = (z_i1, ..., z_i26)
   |                         |
   |                         +--> sigmoid only for thresholded concept metrics
   |
   +--> one linear 26-to-50 species head --> species logits
```

Training minimizes Koh Joint's normalized task-plus-concept loss:

`L = L_task + 0.01 * L_concept`.

`L_task` penalizes wrong species answers. `L_concept` penalizes disagreement
with the 26 supplied concept labels. The factor 0.01 controls their numerical
weight during training; it is not a statement that concepts matter only 1% to
the final prediction.

The class head receives raw `z`; it does not receive probabilities or hard 0/1
concept decisions. There is no learned `1 -> 3 -> 1` concept decoder in this
model. The image encoder is the professor-approved ResNet-50 substitution for
Koh's Inception-v3 encoder. The accepted training description is
`ResNet-50 Koh-architecture Joint CBM, accelerated_v1`, followed by the matched
low-learning-rate convergence continuation recorded in the manifest.
`accelerated_v1` names the declared optimizer, batch, precision, and
learning-rate schedule used to finish training more quickly. It does not replace
the Koh Joint concept bottleneck with an MCBM.

| Symbol | Plain meaning | Use below |
|---|---|---|
| `x_i` | image `i` | model input |
| `y_i` | species label | species-task health |
| `c_ij` | processed 0/1 label for exact concept `j` | concept supervision and health |
| `z_ij` | raw logit emitted for concept `j` | primary grounding quantity |
| `p_ij = sigmoid(z_ij)` | bounded probability | thresholded performance only |
| `c_hat_ij = 1[z_ij>0]` | predicted present/absent concept | recall and balanced accuracy |
| `v_ig` | whether renderer mask `g` is visible | visibility analysis |
| `a_ig` | visible mask area | visibility-strength analysis |

Example: `z_blue_tail=+4` means the model favors “blue tail”; `z_blue_tail=-4`
means it disfavors it. The size of a raw-logit difference is measured in logit
units and is not a probability-point difference.

Ordinary accuracy and recall answer whether the model agrees with labels on
ordinary images. They do **not** establish which pixels produced `z`.


## A new reader's guide: one complete replacement in ordinary language

Suppose the original bird has a **red tail** and we replace only that tail with
a **blue tail** taken from another species.

- The bird receiving the replacement is the **source** bird.
- The species that supplied the blue tail is the **donor**.
- “Red tail” and “blue tail” are two **exact concepts**: specific possible
  values of the broader part “tail.”
- The unchanged picture is the **original**. The otherwise identical picture
  containing the blue tail is the **replacement** or **counterfactual**.
- A **mask** is an image marking which pixels belong to one part. If the blue
  tail mask contains 150 pixels, its visible size is 150 pixels.

The model gives every exact concept an unbounded numerical score called a
**raw logit**, written `z`. Larger means “the model favors this answer more”;
smaller means “it favors it less.” A raw logit is not a percentage. For example,
`z_blue=+4` and `z_red=+1` means blue is favored over red by three logit
units. Applying `sigmoid(z)` produces a probability-like number only when a
thresholded yes/no performance question requires it.

The **margin** compares the two relevant answers:

`margin = blue-tail score - red-tail score`.

- margin `+3`: blue finishes three units above red, so the inserted answer wins;
- margin `-3`: red remains three units above blue, so the old answer wins.

The **response change** (`response_delta`) asks how much that margin moved
toward blue after the pixels changed. Worked example:

1. Before replacement, blue scores `-7` and red scores `+3`, so the starting
   margin is `-7 - 3 = -10`.
2. After replacement, blue scores `+1` and red scores `+2`, so the final
   margin is `+1 - 2 = -1`.
3. The margin moved from `-10` to `-1`, so
   `response_delta = -1 - (-10) = +9`.

The model plainly reacted to the blue pixels because the comparison moved nine
units toward blue, but it still answered red more strongly because the final
margin is negative. That combination—positive response change and negative
final margin—is the report's controlled **backwash event**.

### Other terms used later

| Term | Ordinary meaning | Small example |
|---|---|---|
| rate or fraction | count satisfying a rule divided by all eligible rows | 20 events among 100 swaps gives 0.20 or 20% |
| median | middle value after sorting | the median of 1, 3, 9 is 3 |
| percentile | a location in a sorted distribution | Q95 is greater than or equal to 95% of observed values |
| balanced accuracy | average of success on positive and negative labels | 90% positive recall and 70% negative recall gives 80% |
| visibility bin | replacements grouped by target-part pixel count | 100–199 means the inserted part contains from 100 through 199 pixels |
| label/mask conflict | label says the concept is present while its renderer mask says its pixels are not visible | “red tail=1” but zero red-tail-region pixels |
| exact-value recognition | whether the inserted value receives the largest score among alternatives for that part | blue is highest among nine tail values |
| species support | how many of the 50 species naturally carry an exact value in an unmodified bird; it is not the number of images or swaps | support 4 for blue tail means four species normally have blue tails, even if the experiment renders many blue-tail swaps |
| species decoder | a separate diagnostic classifier trained after the CBM; it asks whether species can be guessed from concept numbers | 70% means 70 of 100 held-out species labels are guessed correctly |
| held-out | rows not used to fit the diagnostic rule being evaluated | fit on four folds and score on the fifth |
| fold | one non-overlapping held-out subset | five-fold testing uses each of five subsets once as the test set |
| RMSE | typical prediction error, with large mistakes penalized more | lower RMSE is better; 3.1 is better than 3.8 |
| residual | what remains after subtracting the comparison group's expected value | observed margin 5 minus expected margin 3 leaves residual +2 |
| association | two measurements vary together; the cause is not isolated | larger visible tails tend to have better margins |
| causal evidence | changing one thing while holding the relevant alternatives fixed changes the outcome | the renderer replaces one part in the same scene |
| grounding | the named concept score actually follows the pixels of that named part | blue-tail score follows replacement blue-tail pixels |
| model health | basic check that an output changes and agrees with ordinary labels | a constant score is unhealthy even if one class is common |
| collapsed output | a score that is effectively identical for every image | always returning `z=2` cannot distinguish presence from absence |
| seed 1 | one fixed random initialization/run identifier | other seeds are independent replications, not extra images in an error bar |
| RLv2 | the later matched model trained after changing positive labels to zero when their part is invisible | used for the causal label test in notebook 02rl, not for the discovery result here |

Figures 3–4 provide causal evidence about the inserted pixels because the
renderer holds the rest of the scene fixed. Later comparisons of visibility,
value frequency, or species are mostly associations: they can identify a
plausible contributor without proving that contributor alone caused the event.


## Investigation map: what would count as backwash?

We begin without assuming that backwash exists. A FunnyBird replacement will
count as a **backwash event** only if both of the following occur on the *same
controlled replacement*:

1. the donor part moves the raw concept comparison toward the donor
   (`response_delta > 0`); and
2. after that movement, the old source concept is still higher
   (`m_cf < 0`).

Numerical example—not a reported result: replacing a red tail with a blue tail
raises the blue-tail score relative to red by 24 units, but red still finishes
6 units above blue. The model reacted to the new tail pixels, yet its final
concept answer remained attached to the old bird. Figure 4 asks whether this
pattern actually appears in the accepted data.

**Part names are outcomes, not mechanisms.** The proposed general mechanism is
competition between the original context-driven source preference and the
response caused by the inserted part pixels. Visibility/label conflict,
exact-value difficulty, alternative frequency, and source-species organization
may change that balance for any part. FunnyBird tail is the most severe observed
example, but all five parts are measured and CUB must establish its own ordering.

The investigation stops or changes direction if an earlier gate fails. It asks:

| Step | Needed fact | Figure(s) | Why it is needed |
|---|---|---|---|
| 1 | the trained concept outputs are usable | 1 | a constant or broken output cannot support grounding analysis |
| 2 | the renderer really changed only the named part | 2 | otherwise a score change cannot be assigned to that part |
| 3 | the inserted pixels cause donorward movement | 3 | proves the model saw some evidence in the new part |
| 4 | starting preference, donor rise, and old-source decrease are separated | 3b | distinguishes starting context from response magnitude |
| 5 | the old source can still win after that movement | 4, 4b | this is the controlled backwash predicate and its complementary outcomes |
| 6 | the event is not a direction-averaging artifact | 5 | checks forward and reverse replacements separately |
| 7 | test proposed contributors | 6–8 | visibility/occlusion, conflicting labels, exact-value difficulty, support/alternatives, and source species |
| 8 | separate species-information availability from controlled-outcome linkage | 8b–8c | prevents calling decodable leakage a causal mechanism |
| 9 | measure what those contributors predict and what remains | 9 | prevents claiming that a plausible story explains all rows |
| 10 | measure downstream class impact | 10 | separates explanation failure from species-classification harm |

### The three contributor hypotheses carried into both reports

The linked comparison tests the same three proposed reasons in the same order:

1. **visibility/occlusion:** the named pixels may be absent or too small;
2. **label–visibility conflict:** training may call a concept positive when its
   mapped region is not visible; and
3. **exact-value difficulty:** some variants may be intrinsically harder, rarer,
   or drawn from a larger alternative set.

Only after those are measured do we ask whether unchanged source species/body
context organizes the remaining raw-score error.  That fourth term is a
residual association, not a promise that the three measured reasons sum to the
whole phenomenon.

The final mechanistic follow-up distinguishes two different statements that
must not be collapsed into one:

1. species can be decoded from the raw concept scores (**information is
   available**);
2. retained source-species evidence on controlled replacements predicts a more
   source-negative final concept margin (**the information is linked to the
   backwash outcome**).

Only the second statement can promote distributed species information from a
leakage diagnostic to a candidate mechanism. Even then, species is bundled with
body shape and pose, so it remains a mechanistic association rather than an
independent species intervention.

The implementation retains the complete renderer audit, all exact values,
species residuals, recall/model-health controls, and provenance inherited from
the earlier curated report and the original renderer-swap and recall notebooks.

### Capabilities and limits that determine this design

FunnyBird supplies an exact renderer mask and a clean donor-part replacement:
body, pose, camera, and background can remain unchanged while one part changes.
That makes Figures 3–4 causal tests of the changed part pixels. Visibility,
training-label conflict, exact value, support, and species are then investigated
as possible contributors. Except for the later matched RLv2 retraining, those
contributor analyses are observational and are not allowed to erase the
controlled event or claim that every cause has been found.

### Predictions stated before the results

- If the concept is locally grounded, replacement should produce
  `response_delta > 0` and usually `m_cf > 0`.
- If backwash occurs, a nontrivial set should have `response_delta > 0` but
  `m_cf < 0`.
- If visibility/occlusion is sufficient, the event should disappear for large,
  clearly visible inserted parts.
- If label–visibility conflict contributes, parts with more positive labels on
  invisible parts should later improve most under matched RLv2 training.
- If exact-value difficulty or species context contributes, matched rows should
  retain systematic value- or species-linked differences.
- None of these predictions requires the measured contributors to reduce the
  remaining error to zero.

### Fast reader path and evidence ladder

The shortest main path is Figures **1 -> 2 -> 3 -> 4 -> 6 -> 7 -> 8 -> 8c ->
9 -> 10**. Figure 8b establishes ordinary-image species-information
availability; Figure 8c is the single focused swap-specific discriminator.
MCBM compression, RLv2 relabelling, and their combination are separate later
chapters and cannot replace the Standard-CBM discovery chain.

| Evidence level | Operation or quantity | Strongest permitted conclusion |
|---|---|---|
| controlled grounding test | replace one named part in the same rendered scene; measure `response_delta` and final `m_cf` | the inserted pixels moved the named comparison, yet the old source sometimes remained higher: controlled FunnyBird backwash |
| contributor test | compare visibility, label/mask conflict, exact-value difficulty, support, and source fingerprint | a factor organizes or predicts failures; it is not automatically an isolated cause |
| matched causal follow-up | retrain the matched RLv2 model after changing only the declared visibility-aware labels | whether that label intervention changes the same fixed-swap outcome |
| CUB/CUB70 approximation | compare natural visible/hidden raw logits, mask conflict, exact-value error, matched species effects, and residuals | whether observational signatures recur in photographs; this is not a donor/source swap and cannot by itself prove CUB backwash |
| future source constraint | add swap-consistency or explicit spatial routing only if the accepted results require it | whether forcing the score to follow the named region reduces the controlled event |

### Relation to existing CBM intervention and leakage work

The original [Concept Bottleneck Models paper](https://proceedings.mlr.press/v119/koh20a.html)
intervenes by editing a predicted concept value before the task head. That tests
whether changing the bottleneck changes the final task output; it does not test
whether image-to-concept prediction used the named pixels. Work on
[whether CBMs learn concepts from the intended input features](https://arxiv.org/abs/2105.04289)
motivates the named-pixel grounding test. Work on
[information leakage in soft CBMs](https://arxiv.org/abs/2211.03656) motivates
the label-versus-raw-score control because soft scores can contain information
beyond their nominal concepts. Spatially aware CBMs explicitly introduce local
concept maps and region editing, illustrating why a spatial constraint is a
different method from merely compressing a scalar bottleneck
([Benou et al., 2025](https://openaccess.thecvf.com/content/CVPR2025/html/Benou_Show_and_Tell_Visually_Explainable_Deep_Neural_Nets_via_Spatially-Aware_CVPR_2025_paper.html)).

Our renderer swap therefore supplies a distinct base-case intervention: it
changes the named pixels first and observes the concept score afterward. The
leakage tests are supporting mechanism diagnostics and cannot replace that
controlled grounding test.


## Dataset design and report population

FunnyBird is synthetic, so the relevant objects are known exactly rather than
estimated from photographs.

| Item | Value used here | Why it matters |
|---|---:|---|
| species | 50 | unchanged species/body appearance is the possible contextual signal |
| named parts | `tail`, `wing`, `beak`, `foot`, `eye` | these are the only five FunnyBird part names used below |
| exact concepts | 26 part values across the five parts | for example, `tail::blue`; a part and its exact value are not interchangeable |
| held-out model-health population | 500 test images | used for Figure 1 and the species decoder |
| controlled swap population | accepted fixed-render seed-1 CSV | the same validated rendered images are reused across model comparisons |

Species determine part values in FunnyBird, so species context can predict a
concept label even when the named part is hard to see. That makes contextual
prediction possible, but it does not prove the trained CBM used context. The
controlled replacement in Figures 2–4 supplies that stronger test.


In [ ]:
import os, json, re, glob, sys, hashlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage, Markdown

CURATED = Path(os.environ["CURATED_DATA"])
CWD = Path.cwd()
REPO = CWD if (CWD/"analysis").is_dir() else CWD.parent
sys.path.insert(0, str(REPO/"data"/"funnybirds"))
plt.rcParams.update({"figure.dpi": 120, "axes.grid": False})
pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", 40)
ORDER = ["tail", "wing", "beak", "foot", "eye"]
COLORS = {"tail":"#6A0DAD", "wing":"#0072B2", "beak":"#E69F00",
          "foot":"#009E73", "eye":"#CC79A7"}

def require(path, command):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}\nProduce it with: {command}")
    return path

MODEL_ROOT = CURATED/"koh_joint_resnet_accelerated_converged_v1"/"funnybirds"/"standard"/"seed1"
SWAP_ROOT = CURATED/"swap_koh_joint_resnet_accelerated_converged_v1_seed1"
MODEL_MANIFEST = require(MODEL_ROOT/"SUCCESS.json", "complete accepted FunnyBird Standard convergence")
SWAP_MANIFEST = require(SWAP_ROOT/"SUCCESS.json", "complete accepted converged FunnyBird fixed swaps")
for manifest_path in [MODEL_MANIFEST, SWAP_MANIFEST]:
    subprocess.run([sys.executable, str(REPO/"analysis"/"canonical_manifest.py"),
                    "verify", "--manifest", str(manifest_path)], check=True)
subprocess.run([sys.executable, str(REPO/"analysis"/"validate_fixed_swaps.py"),
                "--out", str(SWAP_ROOT)], check=True)
model_manifest = json.loads(MODEL_MANIFEST.read_text())
swap_manifest = json.loads(SWAP_MANIFEST.read_text())
expected_model_meta = {"framework":"koh_joint", "backbone":"resnet50",
                       "dataset":"funnybirds", "labels":"standard", "seed":"1"}
for key,value in expected_model_meta.items():
    if model_manifest.get("metadata",{}).get(key) != value:
        raise RuntimeError(f"model manifest {key} is not {value!r}")
if swap_manifest.get("metadata",{}).get("framework") != "koh_joint":
    raise RuntimeError("swap manifest is not Koh Joint")
SWAP = require(SWAP_ROOT/"funnybirds-cbm-s1.csv", "run accepted converged swaps")
S = pd.read_csv(SWAP)
# The Koh Joint model emits these raw concept logits directly. Legacy
# CSV column names are retained only as a file-schema compatibility layer.
if "response_delta" not in S:
    S["response_delta"] = S.margin - (S.z_new_orig - S.z_old_orig)
required_swap_columns={"z_new","z_old","z_new_orig","z_old_orig","margin","response_delta"}
missing_swap_columns=required_swap_columns-set(S.columns)
if missing_swap_columns:
    raise RuntimeError(f"accepted swap CSV is missing {sorted(missing_swap_columns)}")
S["m_orig"] = S.z_new_orig - S.z_old_orig
S["m_cf"] = S.z_new - S.z_old
S["donor_gain"] = S.z_new - S.z_new_orig
S["source_decrease"] = S.z_old_orig - S.z_old
if not np.allclose(S.m_cf,S.margin):
    raise RuntimeError("stored final margin disagrees with z_new-z_old")
if not np.allclose(S.m_cf,S.m_orig+S.donor_gain+S.source_decrease):
    raise RuntimeError("starting-margin/response decomposition does not close")
S["responded_but_source_wins"] = (S.response_delta > 0) & (S.margin < 0)
print("fixed-render input:", SWAP)
print("rows:", len(S), "parts:", sorted(S.part.unique()))

PRED = require(MODEL_ROOT/"final_test.parquet", "complete accepted FunnyBird Standard evaluation")
MODEL = require(MODEL_ROOT/"final_model_1.pth", "complete accepted FunnyBird Standard convergence")
EVAL = pd.read_parquet(PRED)
required_eval_columns={"image","y_true","y_pred","concept_index","concept_name","z","prob","gt_label"}
missing_eval_columns=required_eval_columns-set(EVAL.columns)
if missing_eval_columns:
    raise RuntimeError(f"Koh evaluation is missing {sorted(missing_eval_columns)}")
if len(EVAL) != EVAL.image.nunique()*26:
    raise RuntimeError("Koh evaluation is not one row per image and exact concept")
concept_order=(EVAL[["concept_index","concept_name"]].drop_duplicates()
               .sort_values("concept_index"))
if concept_order.concept_index.tolist() != list(range(26)):
    raise RuntimeError("Koh evaluation concept indices are not exactly 0..25")
image_order=EVAL.image.drop_duplicates().tolist()
z_saved=(EVAL.pivot(index="image",columns="concept_index",values="z")
         .reindex(image_order).to_numpy())
p_saved=(EVAL.pivot(index="image",columns="concept_index",values="prob")
         .reindex(image_order).to_numpy())
c_saved=(EVAL.pivot(index="image",columns="concept_index",values="gt_label")
         .reindex(image_order).to_numpy())
image_labels=(EVAL[["image","y_true","y_pred"]].drop_duplicates("image")
              .set_index("image").reindex(image_order))
y_saved=image_labels.y_true.to_numpy(dtype=int)
y_pred_saved=image_labels.y_pred.to_numpy(dtype=int)
if not np.allclose(1/(1+np.exp(-z_saved)),p_saved,rtol=1e-5,atol=1e-6):
    raise RuntimeError("Koh evaluation probability does not equal sigmoid(raw z)")

FB_ROOT = Path(os.environ.get("FUNNYBIRDS_ROOT", CURATED/"FunnyBirds"))
import funnybirds_concepts as fbc
parts = fbc.load_parts(FB_ROOT)
CONCEPT_NAMES = fbc.concept_names(parts)
SPANS = fbc.group_slices(parts)
if concept_order.concept_name.tolist() != CONCEPT_NAMES:
    raise RuntimeError("FunnyBird concept names/order disagree with the Koh evaluation")
if len(CONCEPT_NAMES) != z_saved.shape[1]:
    raise RuntimeError("parts.json concept width does not match saved predictions")
CONCEPT_PART = {name: part for part,(a,b) in SPANS.items() for name in CONCEPT_NAMES[a:b]}
print("framework: Koh Joint; backbone: ResNet-50; minimal_cbm: rejected")
print("checkpoint:", MODEL)
print("evaluation:", PRED, "images:", len(y_saved), "concepts:", len(CONCEPT_NAMES),
      "species:", len(np.unique(y_saved)))


## 1 · Did training produce a usable, non-collapsed CBM?

**Question.** Did training produce a usable, non-collapsed CBM?

**Variables and prediction.** For every exact concept `j`, measure raw-score spread, positive-versus-negative label separation, balanced accuracy, and positive recall. A usable slot has nonzero spread, positive label separation, and above-chance thresholded performance.

**Method.** Compute all quantities from the accepted converged checkpoint's held-out predictions. Recall is a health statistic, not grounding evidence.

### Figure 1 · Did training produce a usable, non-collapsed CBM?

**How to read the figure.** Each row is one exact concept, such as `yellow tail`. The four panels use the
same rows. `spread = Q95(z)-Q05(z)` asks whether the output changes across
test images; exactly zero means a constant output. `label separation =
median(z|c=1)-median(z|c=0)` asks how far positive-labelled images sit above
negative-labelled images; positive is the expected direction. `balanced
accuracy = (positive recall + negative recall)/2` gives positive and negative
labels equal weight; 0.5 is chance for a binary concept. `positive recall =
P(z>0|c=1)` is the fraction of labelled-positive images called positive.
Example: positive recall 0.90 means 90 of 100 positive-labelled images have
`z>0`. Dot color identifies the FunnyBird part: purple tail, blue wing,
orange beak, green foot, and pink eye. The solid zero line marks no label
separation; the dashed 0.5 lines mark chance balanced accuracy and 50%
positive recall. These are health checks, not evidence about which pixels
produced `z`.


In [ ]:
# ALT: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
def balanced_accuracy(y, pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    z=z_saved[:,j]; c=c_saved[:,j].astype(int); pred=(z>0).astype(int)
    rows.append({"concept":name,"part":CONCEPT_PART[name],
                 "spread":np.quantile(z,.95)-np.quantile(z,.05),
                 "label_separation":np.median(z[c==1])-np.median(z[c==0]),
                 "balanced_accuracy":balanced_accuracy(c,pred),
                 "positive_recall":pred[c==1].mean(),
                 "n_positive":int(c.sum()),"n_negative":int((c==0).sum())})
HEALTH=pd.DataFrame(rows).sort_values(["part","concept"])
y_true=y_saved
task_accuracy=float((y_pred_saved==y_true).mean())
concept_accuracy=float(((z_saved>0)==c_saved).mean())
display(pd.DataFrame([{"images":len(y_true),"species":len(np.unique(y_true)),
                      "task_accuracy":task_accuracy,"concept_accuracy":concept_accuracy}]).round(4))
display(HEALTH.round(3))
metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(15,max(5,.24*len(HEALTH))),sharey=True)
y=np.arange(len(HEALTH))
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=HEALTH.part.map(COLORS).fillna("#BBBBBB"),s=24)
    ax.set_xlabel(m.replace("_"," "))
    if m in ["label_separation"]: ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept,fontsize=7)
axes[0].invert_yaxis(); fig.suptitle("Figure 1 · Exact-concept model-health guard")
plt.tight_layout(); plt.show()


- **Method in one line:** We computed four health statistics directly from the frozen CBM's 26 raw concept logits on 500 held-out images; no classifier or model was fitted.


    ### Plain-language reference for Figure 1

    **Plain caption.** All 26 concept outputs vary and classify their ordinary labels well, so the swap analysis is not being driven by a collapsed or unusable model.

    **Terms and how to read it.** `spread` is the 95th minus 5th percentile of raw `z`; `label separation` is the positive-label median minus the negative-label median; balanced accuracy weights positive and negative recognition equally; positive recall uses positive-labelled images as its denominator.

    **Literal values.** Across 500 held-out images, all 26 exact outputs vary: raw-z spread is 5.907-13.616, label separation is 8.069-12.483, balanced accuracy is 0.969-1.000, and positive recall is 0.940-1.000. Species accuracy is 0.992 and concept accuracy is 0.9968.

    **Interpretation.** The model is not broken or stuck. Every concept score changes across images, and the model almost always agrees with the ordinary labels. That makes the later replacement test meaningful, but it still does not tell us which pixels the model used.

**Strongest alternative explanation.** Excellent label prediction can still come from species context rather than the named part, so this figure establishes health but not grounding.

    **Discriminating test.** Use the same-image controlled replacement in Figures 3-4.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR seed-1 standard-CBM model health; no exact output is collapsed.`

    **Next question.** Is the fixed renderer intervention itself valid?


## 2 · Did the renderer change only the intended part?

**Question.** Did the renderer change only the intended part?

**Variables and prediction.** Inspect the semantic preflight and original/swap/delete/part-map examples for all five parts. For a visible replacement, the target part should change while the rest of the scene is preserved. Rows whose rendered RGB image does not change must remain identifiable and be handled by the later visibility analysis, not counted as visibly changed.

**Method.** Use the accepted full-cache validation plus representative all-part examples before reading any model response.

### Figure 2 · Did the renderer change only the intended part?

**How to read the figure.** Figure 2a is the renderer's semantic preflight: for each part it shows the
original, replacement, deletion, original part map, and replacement part
map. In a visible example, the named part should change while body, pose,
camera, and background remain fixed. A cached row with identical original
and replacement RGB pixels is not called visibly changed; it is retained for
the later exact visibility analysis. This is a pixel-operation gate; it
contains no model result.


In [ ]:
# ALT: FunnyBird semantic renderer preflight showing the intended one-part replacement and deletion for every part.
ROOT = SWAP.parent
preflight_candidates=[ROOT/"renderer_preflight"/"renderer_semantic_preflight.png"]
preflight=next((p for p in preflight_candidates if p.exists()),preflight_candidates[0])
example_candidates=[ROOT/"examples",CURATED/"swap_fixed_v2_attempt2"/"examples"]
examples=next((p for p in example_candidates if p.is_dir()),example_candidates[0])
from PIL import Image
if preflight.exists():
    im0=Image.open(preflight).convert("RGB")
    width,height=im0.size
    fig_width=18
    fig_height=max(8,fig_width*height/width)
    fig0,ax0=plt.subplots(figsize=(fig_width,fig_height),dpi=120)
    ax0.imshow(im0); ax0.axis("off")
    ax0.set_title("Figure 2a · Semantic preflight: original, swap, delete, original map, swap map")
    plt.tight_layout(); plt.show()
else:
    raise FileNotFoundError("accepted converged swap root lacks the semantic preflight sheet")


- **Method in one line:** We displayed the renderer's saved original, one-part replacement, deletion, and part-mask outputs for all five parts; this is a pixel-operation audit with no model fitting.


**Plain caption.** This preflight sheet checks that the renderer can replace
or remove one named part without intentionally changing the remaining bird,
pose, camera, or background.

**How this image was obtained.** Before evaluating the CBM, the renderer
saved, for each part, the original image, the one-part replacement, the
deletion, the original part map, and the replacement part map. No model
score or fitted classifier appears in this figure.

**Literal observation.** Every named part has the required image and mask
roles, and the highlighted map follows the requested part rather than the
whole bird. This is a semantic operation check; Figure 2b makes the
within-row pixel comparison easier to inspect.

**Alternative explanation still open.** A montage cannot prove that every
cached row changed only the intended RGB pixels. The full-file hash and
intervention checks reported around Figure 2b address cache completeness
and diversity, while visible target area is analyzed later.

**Limited conclusion.** KEEP as renderer preflight evidence. It validates
the meaning of the requested operation, not the CBM's response and not the
entire cache by itself.

**Next question.** Do representative stored outputs for all five parts show
the same intended operation clearly enough to inspect row by row?


### Figure 2b · Do saved examples confirm the operation for every part?

**Question.** Does the accepted swap output contain a visually inspectable
original, replacement, deletion, and replacement-part map for tail, wing,
beak, foot, and eye?

**Variables and prediction.** Each row is one named part and each column is
one image role. A valid visible example changes the named part and its map
while leaving the remaining bird and scene unchanged. Across the complete
cache, 98.3% of replacement RGB images differ from their original; the
remaining 1.7% are retained for the later visibility analysis rather than
described as visibly changed. “Missing” is an error, not evidence.

**Method.** Select the first stored audit example by filename order for each
part. This is a complete five-part semantic check, not a hand-picked model
success/failure gallery.

**How to read the figure.** Compare columns within a row, then compare the
visible changed pixels with the highlighted replacement-part map. No axis or
color encodes a model score.


In [ ]:
# ALT: Complete five-part FunnyBird intervention audit showing original, replacement, deletion, and replacement-part map.
ROOT = SWAP.parent
example_candidates=[ROOT/"examples"]
examples=next((p for p in example_candidates if p.is_dir()),example_candidates[0])
if not examples.is_dir():
    raise FileNotFoundError("accepted converged swap root lacks example images")
tags=["orig","swap","delete","target_mask"]
segmentation_colors={"beak":(255,255,0),"eye":(255,255,253),
                     "wing":(0,255,1),"foot":(255,0,1),"tail":(0,0,255)}
fig,axes=plt.subplots(len(ORDER),len(tags),figsize=(12,13))
for r,part in enumerate(ORDER):
    for c,tag in enumerate(tags):
        ax=axes[r,c]
        file_tag="swap_partmap" if tag=="target_mask" else tag
        files=sorted(examples.glob(f"{part}_*_{file_tag}.png"))
        if files and tag=="target_mask":
            segmentation=np.asarray(Image.open(files[0]).convert("RGB"))
            target=np.all(segmentation==np.asarray(segmentation_colors[part]),axis=2)
            ax.imshow(target,cmap="gray",vmin=0,vmax=1)
        elif files:
            ax.imshow(Image.open(files[0]).convert("RGB"))
        else:
            ax.text(.5,.5,"missing",ha="center",va="center")
        ax.set_title(f"{part} · {tag}"); ax.axis("off")
fig.suptitle("Figure 2b · Representative five-part intervention audit")
plt.tight_layout(); plt.show()


- **Method in one line:** We selected one accepted saved audit row per part and displayed its original, replacement, deletion, and isolated target mask; no result was estimated from these examples.


    ### Plain-language reference for Figure 2b

    **Plain caption.** The renderer targets the named part while preserving the rest of the bird and scene; 98.3% of cached replacements visibly change RGB pixels, while the remainder are retained for visibility analysis.

    **Terms and how to read it.** Rows are named parts; columns are original, replacement, deletion, and part-map roles. The colors are image pixels or renderer masks, not model scores.

    **Literal values.** For tail, wing, beak, foot, and eye, the displayed replacement and deletion alter the named part while the body, pose, camera, and background remain fixed; the target part map contains that region. Across the complete cache, 98.3% of replacement RGB images differ visibly from their originals. The remaining 1.7% are retained and identified by the later pixel-visibility measurement rather than described as visibly changed.

    **Interpretation.** The pictures and full-file checks agree that the operation targets the named part rather than silently replacing the whole bird or scene. Most cached replacements visibly change RGB pixels; the small unchanged group is kept and measured later instead of being treated as a visible intervention.

**Strongest alternative explanation.** The five displayed rows are representative semantic checks, not by themselves proof about every cached image.

    **Discriminating test.** Retain the semantic preflight plus the accepted fixed-render hash/diversity validation across all evaluated models and render IDs.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR the validated FunnyBird fixed-render intervention, with visibly unchanged rows retained explicitly for the visibility analysis.`

    **Next question.** Do those inserted pixels move the raw concept comparison toward the donor?


## 3 · Did the inserted pixels move the comparison toward the donor?

**Question.** Did the inserted pixels move the comparison toward the donor?

**Variables and prediction.** `response_delta = (z_donor-z_source)_cf - (z_donor-z_source)_orig`. Legacy CSV columns named `z_*` contain these post-head raw logits. Values above zero mean that replacement pixels moved the model toward the donor concept.

**Method.** Plot the complete distribution for every part and report the positive-response rate.

### Figure 3 · Did the inserted pixels move the comparison toward the donor?

**How to read the figure.** Panel A puts part on the x-axis and `response_delta` in raw-logit units on the
y-axis. The box spans the 25th--75th percentiles, the orange line is the
median, and whiskers are the 5th--95th percentiles; outliers are omitted only
from drawing. Zero means no donorward change and values above zero mean the
donor gained relative to the old source. Panel B reports the fraction above
zero, with `n` printed over each bar. Colors identify parts using the shared
FunnyBird palette. This measures response size, not whether the donor wins.
Example: a margin change from -20 before replacement to -5 afterward gives
`response_delta=+15`, although the final margin remains negative.


In [ ]:
# ALT: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
vals=[S.loc[S.part==p,"response_delta"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("response_delta (raw logit units)")
axes[0].set_title("A · Distribution of donorward movement")
rate=S.groupby("part").response_delta.apply(lambda x:(x>0).mean()).reindex(ORDER)
axes[1].bar(rate.index,rate.values,color=[COLORS[p] for p in rate.index])
axes[1].set_ylim(0,1.05)
axes[1].set_ylabel("fraction with response_delta > 0"); axes[1].set_title("B · Positive donor-response rate")
counts=S.groupby("part").size().reindex(ORDER)
for x,(part,value) in enumerate(rate.items()):
    axes[1].text(x,value+.02,f"n={int(counts.loc[part])}",ha="center",fontsize=8)
fig.suptitle("Figure 3 · Does the replacement produce the predicted within-image response?")
plt.tight_layout(rect=[0,0,1,.94]); plt.show()
display(pd.DataFrame({"eligible_swaps":counts,"positive_response_rate":rate}).round(3))


- **Method in one line:** For each of 5,000 paired swaps, we subtracted the original donor-minus-source margin from the counterfactual margin, then summarized those paired raw-logit changes by part; nothing was trained.


    ### Plain-language reference for Figure 3

    **Plain caption.** Inserted part pixels move the donor-versus-source comparison toward the donor for nearly every swap, although this does not yet say which concept wins.

    **Terms and how to read it.** `response_delta=m_cf-m_orig` is measured in raw-logit units. Above zero means donorward movement. The box shows the middle 50%, whiskers show the 5th--95th percentiles, and each rate uses all 1,000 swaps for that part.

    **Literal values.** The complete response distributions are donorward for nearly every swap. Positive-response rates are tail 0.919, wing 1.000, beak 0.989, foot 0.997, and eye 0.986, with 1,000 swaps per part.

    **Interpretation.** The model nearly always notices the new part: its answer moves toward the inserted value in at least 91.9% of swaps for every part. The next question is whether that movement is large enough to change the final answer.

**Strongest alternative explanation.** A positive movement alone does not say that the inserted donor finishes above the old source.

    **Discriminating test.** Inspect the final donor-minus-source margin jointly with response_delta.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR a causal within-image response of the controlled part-replacement intervention in all five part groups. This is a part-level statement: the observed positive-response rates are 0.919-1.000, not a claim that every individual row responded. Tail, beak, and eye have smaller typical movement than wing and foot; tail is not the mechanism and is not the only comparatively weak response.`

    **Next question.** Did the parts start equally far behind, and did donor rise versus source release contribute differently?


## 3b — Where did each part start, and which score changed after replacement?

**Question.** Does a part finish poorly because its donor began far below
the source, because the donor rose too little, because the removed source
fell too little, or because several of these occurred together?

**Variables and exact identity.** For every swap:

`m_orig = z_donor,orig - z_source,orig`

`donor_gain = z_donor,cf - z_donor,orig`

`source_decrease = z_source,orig - z_source,cf`

`response_delta = donor_gain + source_decrease`

`m_cf = m_orig + response_delta`

**Score scale.** Every quantity here uses the post-head raw logit
`z=q(h)`, which is unbounded. This standard CBM has no MCBM gamma penalty
and no `±3` target. Notebook 03 applies the soft `±3` target to internal
`h`, not to the plotted `z`.

`m_orig` is the starting preference on the unchanged original image. It
is **not** a pure context measurement because the source part is still
visible there. Species/body context is tested separately later.

Example: the donor starts 20 units below the source, then rises by 9 while
the old source falls by 6. Total donorward response is 15, so the final
margin is `-20+9+6=-5`: the swap helped, but the source still wins.

**Method.** Average each raw-logit quantity over all 1,000 validated swaps
for each part, including both directions. Verify the exact row-wise
identity before displaying any mean.

### Figure 3b — Starting preference, donor rise, source release, response, and final result

**How to read the figure.** All panels use the same raw-logit y-axis and
part colors. Panel A below zero means the future donor starts behind.
Panels B and C above zero are the two ways replacement helps. Panel D is
their sum. Panel E above zero means the donor finally wins. Part names
identify observed outcomes, not mechanisms.


In [ ]:
# ALT: Standard FunnyBird CBM starting margin, donor-score gain, removed-source decrease, total response, and final margin for all five parts.
component_columns=["m_orig","donor_gain","source_decrease","response_delta","m_cf"]
component_means=S.groupby("part")[component_columns].mean().reindex(ORDER)
decomposition_error=np.max(np.abs(S.m_cf-(S.m_orig+S.donor_gain+S.source_decrease)))
if decomposition_error>1e-8: raise RuntimeError(f"decomposition error {decomposition_error}")
titles=[("m_orig","A. Before swap: donor minus source"),
        ("donor_gain","B. Inserted donor score rises"),
        ("source_decrease","C. Removed source score falls"),
        ("response_delta","D. Total donorward movement"),
        ("m_cf","E. After swap: donor minus source")]
lim=float(np.nanmax(np.abs(component_means.values)))*1.12
fig,axes=plt.subplots(1,5,figsize=(21,4.4),sharey=True)
for ax,(column,title) in zip(axes,titles):
    values=component_means[column]
    ax.bar(ORDER,values.values,color=[COLORS[p] for p in ORDER],alpha=.75)
    ax.axhline(0,color="black",lw=.9); ax.set_title(title,fontsize=10)
    ax.tick_params(axis="x",rotation=45); ax.set_ylim(-lim,lim)
axes[0].set_ylabel("mean raw-logit units")
fig.suptitle("Figure 3b — What creates each part's final donor-versus-source result?")
plt.tight_layout(); plt.show(); display(component_means.round(3))
print("maximum row-wise decomposition error:",decomposition_error)


- **Method in one line:** We arithmetically decomposed every paired swap into starting margin, donor-score rise, source-score fall, total response, and final margin, verified the identity row by row, and averaged each term by part.


### Plain-language reference for Figure 3b

**Plain caption.** The final donor-versus-source result combines a
starting preference for the source with the donor's rise and the
source's fall after replacement.

**Terms.** Starting margin is donor score minus source score before
replacement. Inserted donor score rises is the increase in the donor
concept's score. Removed source score falls is the decrease in the old
source concept's score. Total donorward movement is donor rise plus
source decrease. Final margin is starting margin plus total movement. A
negative final margin means the old source concept remains higher. All
five panels use mean raw-logit units over 1,000 swaps per part.

**Literal mean values.**

| Part | Starting margin | Donorward movement | Final margin |
|---|---:|---:|---:|
| tail | `-10.756` | `+9.800` | `-0.956` |
| wing | `-9.914` | `+16.390` | `+6.476` |
| beak | `-9.167` | `+11.778` | `+2.611` |
| foot | `-8.847` | `+13.831` | `+4.984` |
| eye | `-8.481` | `+11.749` | `+3.268` |

The row-wise arithmetic identity closes to numerical error below
`1.8e-15`. Tail's donor rise is `4.599` and old-source decrease is
`5.201`, both the smallest part means.

**Interpretation.** Every part starts with a strong source preference.
Wing, beak, foot, and eye usually produce enough movement to overcome
it. Tail produces a large response too, but its average response is
insufficient to erase the starting preference. Tail does not mainly
fail because it began uniquely far behind; its total correction is
smaller.

**Alternative.** Tail could be smaller, hidden more often, harder to
distinguish exactly, or more strongly associated with species context.
Means can also hide direction and exact-value asymmetry, and the original
margin still contains genuine source-part pixels.

**Discriminating test.** Separate direction, visibility, exact values,
and source-species organization in the following figures.

**Verdict.** **KEEP**.

**Proof ledger.** The competition producing the final margin is
separated arithmetically. The causes of the starting preference and
unequal replacement response remain unresolved.

**Next question.** How often does the donor actually finish higher?


## 4 · After responding, does the donor finish above the old source?

**Question.** After responding, does the donor finish above the old source?

**Variables and prediction.** The final margin is `m_cf=z_donor,cf-z_source,cf`. The primary event is `response_delta>0` with `m_cf<0`. A lower-right quadrant point means the inserted pixels had an effect but the old source still wins.

**Method.** Show final-margin distributions and the joint response/margin plane for every part.

### Figure 4 · After responding, does the donor finish above the old source?

**How to read the figure.** In the margin panel, zero separates donor wins (`m_cf>0`) from old-source wins
(`m_cf<0`). In the quadrant panel, x is donorward movement and y is the final
donor-minus-source score. The lower-right quadrant is the controlled
backwash predicate `response_delta>0 and m_cf<0`: the new pixels moved the
answer toward the donor, but the old source still finished higher. Boxes and colors use the Figure 3 definitions;
translucent points are individual swaps and the legend maps color to part.
Example: `m_cf=-5` means the old source finishes five raw-logit units above
the donor.


In [ ]:
# ALT: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
fig,axes=plt.subplots(1,2,figsize=(14,4.8))
vals=[S.loc[S.part==p,"margin"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("final margin m_cf (donor − source)")
axes[0].set_title("A · Final donor-minus-source margin")
for p in ORDER:
    d=S[S.part==p]
    axes[1].scatter(d.response_delta,d.margin,s=10,alpha=.22,color=COLORS[p],label=p)
axes[1].axvline(0,color="black",lw=1); axes[1].axhline(0,color="black",lw=1)
axes[1].set_xlabel("response_delta"); axes[1].set_ylabel("final margin m_cf")
axes[1].set_title("B · Lower-right = responds, but old source still wins")
axes[1].legend(ncol=5,fontsize=8)
fig.suptitle("Figure 4 · Controlled FunnyBird backwash predicate")
plt.tight_layout(); plt.show()
summary=S.groupby("part").agg(n=("margin","size"),median_response=("response_delta","median"),
    median_final_margin=("margin","median"),positive_response_rate=("response_delta",lambda x:(x>0).mean()),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER)
display(summary.round(3))


- **Method in one line:** We applied the predeclared row-level predicate `response_delta>0 and m_cf<0` to all 1,000 validated swaps per part and plotted the two raw quantities jointly; no threshold was learned.


    ### Plain-language reference for Figure 4

    **Plain caption.** Controlled backwash is graded: the model responds to the new pixels but the old source answer still wins most often for tail and less often for every other part.

    **Terms and how to read it.** `m_cf=z_donor,cf-z_source,cf`. A negative final margin means the old source remains higher. The controlled event requires both `response_delta>0` and `m_cf<0` on the same row.

    **Literal values.** Median final margins are tail -0.819, wing 6.483, beak 2.551, foot 5.158, and eye 3.511 raw-logit units. On the same 1,000 swaps per part, the donorward-response-but-source-wins rates are 0.502, 0.019, 0.200, 0.032, and 0.089, respectively.

    **Interpretation.** Yes, backwash occurs. Tail is the clearest case: in about half the tail replacements, the model reacts in the correct direction but still favors the tail value belonging to the original bird. The same event also occurs less often for beak, eye, foot, and wing.

**Strongest alternative explanation.** Starting preference, swap direction, target visibility, exact value difficulty, and source species could organize the unequal rates.

    **Discriminating test.** Test those alternatives separately in Figures 5-9 without changing the event definition.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR the seed-1 controlled FunnyBird backwash predicate across a graded part ordering: strongest for tail, beak, and eye, with minority events also in wing and foot. This is not a tail-specific mechanism claim.`

    **Next question.** Can swap direction create the pooled pattern?


## 4b — How often does the donor win, help but still lose, or fail to move donorward?

Every validated swap is placed into exactly one outcome:

1. `m_cf > 0`: the donor concept finishes higher;
2. `m_cf <= 0 and response_delta > 0`: the new pixels help, but the old
   source concept remains higher;
3. `m_cf <= 0 and response_delta <= 0`: the source remains higher and the
   replacement does not move the comparison toward the donor.

These fractions sum to one for every part. Thus Figure 4's controlled-
backwash rate is not the donor-win rate.

Example: if 20 of 100 swaps end donor-positive, 50 move donorward but
remain source-negative, and 30 do not move donorward, the three displayed
fractions are 0.20, 0.50, and 0.30. The denominator is all 100 swaps.

### Figure 4b — Three mutually exclusive outcomes for every part

**How to read the figure.** Every panel contains all five parts and uses a
fraction from zero to one. Higher is desirable only in Panel A. Panel B
is the controlled backwash event. Panel C is a different failure: no
positive response. Both swap directions are included. Bar colors use the
shared part palette defined at the start of the notebook.


In [ ]:
# ALT: Standard FunnyBird CBM donor-win, donorward-but-source-still-wins, and no-donorward-movement fractions for all five parts.
outcomes=pd.DataFrame(index=ORDER,dtype=float)
outcomes["donor_wins"]=(S.m_cf>0).groupby(S.part).mean().reindex(ORDER)
outcomes["helped_but_source_wins"]=((S.m_cf<=0)&(S.response_delta>0)).groupby(S.part).mean().reindex(ORDER)
outcomes["no_donorward_move_and_source_wins"]=((S.m_cf<=0)&(S.response_delta<=0)).groupby(S.part).mean().reindex(ORDER)
if not np.allclose(outcomes.sum(axis=1).values,1):
    raise RuntimeError("three outcome fractions do not sum to one")
panels=[("donor_wins","A. Donor finishes higher"),
        ("helped_but_source_wins","B. New pixels help, but source stays higher"),
        ("no_donorward_move_and_source_wins","C. No donorward movement; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(15,5.4),sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    values=outcomes[column]
    y=np.arange(len(ORDER))
    ax.barh(y,values.values,color=[COLORS[p] for p in ORDER],alpha=.78)
    ax.set_title(title,fontsize=10); ax.set_xlim(0,1.08)
    ax.set_yticks(y,ORDER); ax.invert_yaxis()
    for yy,value in enumerate(values): ax.text(value+.015,yy,f"{value:.3f}",va="center",fontsize=8)
    ax.set_xlabel("fraction of all swaps")
fig.suptitle("Figure 4b — Final outcome categories for standard CBM")
plt.tight_layout(rect=[0,0,1,.94]); plt.show(); display(outcomes.round(3))


- **Method in one line:** We assigned every swap to exactly one of three predeclared outcomes from the signs of `m_cf` and `response_delta`, then divided each count by all 1,000 swaps for that part.


### Plain-language reference for Figure 4b

**Plain caption.** Every replacement is assigned to exactly one final
outcome, separating successful donor wins from insufficient donorward
corrections and from complete failures to move donorward.

**Terms and denominator.** Donor wins means `m_cf>0`. Helped but source
wins means `response_delta>0` and `m_cf<=0`, the controlled backwash
event. No donorward move means both quantities are non-positive. Each
fraction uses all 1,000 swaps for that part, and the three bars sum to one.

**Literal values.** Donor-win/helped-but-source-wins/no-donorward-move
fractions are tail `0.417/0.502/0.081`, wing `0.981/0.019/0.000`, beak
`0.789/0.200/0.011`, foot `0.965/0.032/0.003`, and eye
`0.900/0.089/0.011`.

**Interpretation.** Tail usually notices the replacement: only 8.1% of
tail swaps fail to move toward the donor. The larger problem is that the
correction is insufficient—50.2% move the right way but retain the old
answer. Most beak and eye failures have the same form.

**Alternative.** A positive response may be tiny, and pooled outcomes
may hide direction or exact-value asymmetry.

**Discriminating test.** Retain Figure 3b's response magnitudes and next
separate directions and exact donor values.

**Verdict.** **KEEP**.

**Proof ledger.** The controlled event is distinguished from a model
that simply did not react to the inserted pixels.

**Next question.** Does the pattern occur in both swap directions?


## 5 · Could opposite swap directions create the result?

**Question.** Could opposite swap directions create the result?

**Variables and prediction.** Compare forward and backward rates of `response_delta>0 and final margin<0`, together with median margins. A genuine part pattern should appear in both directions rather than cancel when pooled.

**Method.** Keep directions separate and show their denominators.

### Figure 5 · Could opposite swap directions create the result?

**How to read the figure.** Each part has separate forward (`fwd`) and backward (`bwd`) replacement
estimates, shown as unconnected circles and squares. The rate is
the fraction of rows in the lower-right quadrant from Figure 4; the printed
denominator is the number of swaps. Similar values in both directions argue
against a pooled average hiding opposite effects. A rate of 0.60 means 60%
of swaps in that direction satisfy both `response_delta>0` and `m_cf<0`.


In [ ]:
# ALT: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
D=(S.groupby(["part","direction"]).agg(n=("margin","size"),median_margin=("margin","median"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(len(ORDER))
for direction,marker,offset in [("fwd","o",-.10),("bwd","s",.10)]:
    d=D[D.direction==direction].set_index("part").reindex(ORDER)
    axes[0].scatter(x+offset,d.responded_but_source_wins_rate,marker=marker,label=direction,s=45)
    axes[1].scatter(x+offset,d.median_margin,marker=marker,label=direction,s=45)
    for k,part in enumerate(ORDER):
        axes[0].annotate(f"n={int(d.loc[part,'n'])}",
                         (x[k]+offset,d.loc[part,"responded_but_source_wins_rate"]),
                         xytext=(0,7 if direction=="fwd" else -12),
                         textcoords="offset points",ha="center",fontsize=6)
for ax in axes: ax.set_xticks(x,ORDER)
axes[0].set_ylim(0,1); axes[0].set_ylabel("fraction: donorward response, but source still wins")
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("median final margin")
axes[0].legend(); axes[1].legend(); fig.suptitle("Figure 5 · Forward and backward directions")
plt.tight_layout(); plt.show(); display(D.round(3))


- **Method in one line:** We split each part's 1,000 fixed swaps into its 500 forward and 500 backward replacements and recomputed the same controlled-event rate and final-margin summary independently; nothing was fitted.


    ### Plain-language reference for Figure 5

    **Plain caption.** The part ordering appears in both replacement directions rather than being created by pooling one easy and one difficult direction.

    **Terms and how to read it.** Forward and backward name the two replacement directions. Every displayed rate is controlled-event rows divided by the 500 swaps in that direction; markers are not connected because direction is categorical.

    **Literal values.** Forward and backward results preserve the ordering. Tail rates are 0.528 and 0.476; beak is 0.200 in both; eye is 0.090 and 0.088; foot is 0.022 and 0.042; wing is 0.010 and 0.028. Each direction has 500 swaps.

    **Interpretation.** The ordering is not created by averaging an easy direction with a hard direction. Replacing A with B and replacing B with A give similar part rankings, especially for tail, beak, and eye.

**Strongest alternative explanation.** Individual source/donor value pairs can still be asymmetric even when pooled directions agree.

    **Discriminating test.** Inspect every exact inserted value and both direction-specific denominators.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR excluding opposite-direction cancellation as the main explanation.`

    **Next question.** How does exact target visibility change the event?


## 6 · How much of the result is associated with target visibility?

**Question.** How much of the result is associated with target visibility?

**Variables and prediction.** Use `pixel_count_cf` from the exact swapped-part map and the same final-margin and `response_delta>0, margin<0` definition. If visibility is sufficient, highly visible replacements should remove the part gap; a remaining gap requires another explanation.

**Method.** Use declared bins and print the number of swap rows in every bin.

### Figure 6 · How much of the result is associated with target visibility?

**How to read the figure.** The x-axis bins swaps by the number of visible pixels in the inserted target
part. One panel shows median final raw-logit margin; the other shows the
responded-but-source-wins fraction. If visibility were the whole explanation,
sufficiently large visible parts should make margins positive and drive that
fraction near zero for every part. Point color identifies part; the table
gives the exact denominator for every nonempty bin. The companion visible-only
summary uses the same rule for all parts: `pixel_count_cf > 0`. A median
margin of +3 means the donor finishes three raw-logit units above the source.
Example: a replacement with 120 target-part pixels enters the `100--199`
bin; binning records visibility already present in the render and does not
add pixels to the image or information to the CBM.


In [ ]:
# ALT: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
if "pixel_count_cf" not in S: raise RuntimeError("fixed swap CSV lacks pixel_count_cf")
bins=[0,20,50,100,200,500,np.inf]; labels=["0–19","20–49","50–99","100–199","200–499","500+"]
V=S.copy(); V["visibility_bin"]=pd.cut(V.pixel_count_cf,bins=bins,labels=labels,right=False)
T=V.groupby(["part","visibility_bin"],observed=True).agg(
    n=("margin","size"),median_margin=("margin","median"),responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index()
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for p in ORDER:
    d=T[T.part==p].set_index("visibility_bin").reindex(labels)
    axes[0].plot(labels,d.median_margin,"o",label=p,color=COLORS[p])
    axes[1].plot(labels,d.responded_but_source_wins_rate,"o",label=p,color=COLORS[p])
    for k,label in enumerate(labels):
        if pd.notna(d.loc[label,"n"]):
            axes[0].annotate(f"n={int(d.loc[label,'n'])}",(k,d.loc[label,"median_margin"]),
                             xytext=(2,5),textcoords="offset points",fontsize=5,color=COLORS[p])
axes[0].axhline(0,color="black",lw=.8); axes[0].set_ylabel("median final margin")
axes[1].set_ylim(0,1); axes[1].set_ylabel("fraction: donorward response, but source still wins")
for ax in axes: ax.tick_params(axis="x",rotation=45); ax.legend(fontsize=8,ncol=2)
fig.suptitle("Figure 6 · Same-render visibility analysis")
VISIBLE_ONLY=(V[V.pixel_count_cf>0].groupby("part").agg(
    n_visible_rows=("margin","size"),median_margin=("margin","median"),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER))
plt.tight_layout(); plt.show(); display(T.round(3)); display(VISIBLE_ONLY.round(3))


- **Method in one line:** We grouped the same fixed swaps by the number of visible inserted-part mask pixels and recomputed median final margin and controlled-event fraction inside each declared bin; the images and CBM were unchanged.


    ### Plain-language reference for Figure 6

    **Plain caption.** Greater target-part visibility usually helps, but clearly visible replacements still leave controlled backwash events, especially for tail.

    **Terms and how to read it.** The x-axis is visible target-mask pixels after replacement. The left outcome is median final margin; the right outcome is the controlled-event fraction. Every table row prints its own denominator.

    **Literal values.** Visibility helps but is not sufficient. Tail's median margin changes from -0.819 over all rows to 0.057 for any visible target and 1.416 for targets with at least 100 pixels, while its event rate remains 0.372 in that clear-visibility population. Beak and eye rates generally fall with visibility; tail is non-monotone and its 500+ bin has only 23 rows.

    **Interpretation.** Making the inserted part clearly visible helps, especially for tail, beak, and eye. It does not solve the problem: even among clearly visible tail replacements, roughly 37 of every 100 still react toward the donor but finish with the old source answer higher. The visibility bins contain different exact values and species, so small non-monotonic steps between neighboring bins are not evidence that extra pixels themselves hurt.

**Strongest alternative explanation.** Pixel count is associated with pose, source/donor value, and species, so bins do not isolate visibility causally by themselves.

    **Discriminating test.** Hold exact values and species fixed, and test the visibility-aware label change later with matched RLv2 training.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR visibility as a contributor, not a sufficient explanation.`

    **Next question.** Did ordinary training assign positive labels when the part was not visible?


## 6b · How often did the original training label conflict with visible part evidence?

**Question.** How often did the original training label conflict with visible part evidence?

**Variables and prediction.** Compare the standard and visibility-aware label views for every image used in final training (train plus validation); count positive concept labels changed to zero within each exact concept and part group. A large conflict count identifies a plausible training signal that can reward contextual prediction, but its causal effect belongs to notebook 02rl.

**Method.** Require identical ordered image/class records in both splits and allow only `attribute_label` to differ. This cell compares data labels, not Standard and RLv2 model predictions.

### Figure 6b · How often did the original training label conflict with visible part evidence?

**How to read the figure.** Each row is one exact concept. The x-axis is
`P(visibility-aware label=0 | original label=1)`: the number of original
positive training labels removed by the visibility rule divided by all
original positive labels for that concept. A value of 0.25 means 25 of 100
positive labels conflict with visible part evidence. Color identifies part.
This is a data rate, not a model probability or causal model effect.


In [ ]:
# ALT: FunnyBird training-image counts whose positive part-concept labels change under the matched visibility-aware relabeling rule.
import pickle
standard_input=CURATED/"koh_joint_inputs"/"funnybirds"/"standard"
visibility_input=CURATED/"koh_joint_inputs"/"funnybirds"/"rlv2"
pairs=[]
for split in ["train","val"]:
    std_path=standard_input/f"{split}.pkl"
    visibility_path=visibility_input/f"{split}.pkl"
    if not (std_path.exists() and visibility_path.exists()):
        raise RuntimeError(f"missing matched {split} label views: {std_path} or {visibility_path}")
    std=pickle.loads(std_path.read_bytes())
    visibility=pickle.loads(visibility_path.read_bytes())
    if len(std)!=len(visibility):
        raise RuntimeError(f"standard/visibility-aware {split} lengths differ")
    pairs.extend((split,a,b) for a,b in zip(std,visibility))
positive=np.zeros(len(CONCEPT_NAMES),dtype=int); changed=np.zeros(len(CONCEPT_NAMES),dtype=int)
split_rows=[]
for split in ["train","val"]:
    split_positive=np.zeros(len(CONCEPT_NAMES),dtype=int)
    split_changed=np.zeros(len(CONCEPT_NAMES),dtype=int)
    for _,a,b in [row for row in pairs if row[0]==split]:
        for key in a:
            if key=="attribute_label": continue
            av,bv=a[key],b[key]
            equal=np.array_equal(np.asarray(av),np.asarray(bv)) if isinstance(av,(list,tuple,np.ndarray)) else av==bv
            if not bool(equal): raise RuntimeError(f"non-label record field differs in {split}: {key}")
        ca=np.asarray(a["attribute_label"]); cb=np.asarray(b["attribute_label"])
        split_positive += (ca==1); split_changed += ((ca==1)&(cb==0))
    positive += split_positive; changed += split_changed
    split_rows.append({"split":split,"images":sum(row[0]==split for row in pairs),
                       "positive_labels":int(split_positive.sum()),
                       "positive_to_zero_conflicts":int(split_changed.sum())})
CONFLICT_EXACT=pd.DataFrame({"concept":CONCEPT_NAMES,"part":[CONCEPT_PART[n] for n in CONCEPT_NAMES],
    "n_positive":positive,"n_changed":changed})
CONFLICT_EXACT["conflict_rate"]=CONFLICT_EXACT.n_changed/CONFLICT_EXACT.n_positive.replace(0,np.nan)
PART_CONFLICT=(CONFLICT_EXACT.groupby("part").agg(n_positive=("n_positive","sum"),
    n_changed=("n_changed","sum")).reindex(ORDER))
PART_CONFLICT["conflict_rate"]=PART_CONFLICT.n_changed/PART_CONFLICT.n_positive
q=CONFLICT_EXACT.sort_values(["part","concept"]); y=np.arange(len(q))
fig,ax=plt.subplots(figsize=(10,max(6,.24*len(q))))
ax.barh(y,q.conflict_rate,color=q.part.map(COLORS)); ax.set_yticks(y,q.concept,fontsize=7)
ax.invert_yaxis()
conflict_axis_max=max(.05,min(1.0,float(q.conflict_rate.max())*1.15))
ax.set_xlim(0,conflict_axis_max)
ax.set_xlabel("fraction of positive training labels removed by visibility rule")
ax.set_title("Figure 6b · Exact-concept label/mask conflict in train + validation")
plt.tight_layout(); plt.show()
print("Figure 6b denominators by split:")
display(pd.DataFrame(split_rows))
print("Figure 6b exact-concept and part totals:")
display(q.round(3)); display(PART_CONFLICT.round(3))


- **Method in one line:** We joined the Standard and visibility-aware train-plus-validation label records image by image and counted only positive labels changed to zero by the visibility rule; this is a data audit, not a model comparison.


    ### Plain-language reference for Figure 6b

    **Plain caption.** Original supervision frequently marks a tail concept present when its renderer pixels are invisible, while this conflict is rare for wing and foot.

    **Terms and how to read it.** The numerator is original positive labels changed to zero by the visibility rule; the denominator is all original positive labels for that concept or part. This is a data-conflict rate, not a model probability.

    **Literal values.** Across the 45,000 training and 5,000 validation images, the visibility rule removes 8,184 of 188,461 positive labels. By part, it removes 7,489/37,707 tail labels (0.199), 367/37,617 beak (0.010), 268/37,726 eye (0.007), 48/37,723 foot (0.001), and 12/37,688 wing (less than 0.001).

    **Interpretation.** The ordinary training labels often say a tail value is present when the tail pixels are not visible. This happens for about one tail label in five but is almost absent for wing and foot. Such supervision could teach the model to infer tail from the rest of the bird; notebook 02rl tests that causal proposal by changing the labels and retraining.

**Strongest alternative explanation.** These are training-signal counts, not measured causal effects on the trained standard model.

    **Discriminating test.** Compare otherwise matched standard and RLv2 checkpoints on the same fixed renders.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR a measured, part-specific label/visibility conflict. It is extremely large for tail, small but nonzero for beak/eye, and near zero for wing/foot. This can explain tail's excess severity but cannot by itself explain backwash in every part; causal credit remains deferred to notebook 02rl.`

    **Next question.** Are some exact visual variants much harder than others?


## 7 · Do exact source and donor values explain the failures?

**Question.** Do exact source and donor values explain the failures?

**Variables and prediction.** For every part, compare the inserted donor value with the concept value that has the largest post-swap raw score. A clean diagonal means exact visual values are distinguished; recurring bright columns indicate default answers.

**Method.** Display all parts and all values with row-normalized counts.

### Figure 7 · Do exact source and donor values explain the failures?

**How to read the figure.** Each heatmap row is the value actually inserted and each column is the value
with the largest post-swap raw logit. A bright diagonal means the model names
the inserted value; bright off-diagonal cells show systematic confusion.
Every FunnyBird part and every value is included. The lower row gives the
final-margin distribution for the same inserted values, with the number of
swaps printed above each box. Thus recognition and retained-source margin are
visible together rather than inferred from a diagonal rate alone. A diagonal
value of 0.80 means the inserted value is highest in 80% of that row's swaps.


In [ ]:
# ALT: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
available=[p for p in ORDER if any(c.startswith(f"z_cf_{p}_") for c in S.columns)]
if set(available)!=set(ORDER): raise RuntimeError(f"missing all-part post-swap concept logits: have {available}")
fig,axes=plt.subplots(2,5,figsize=(20,9.5),constrained_layout=True)
diag={}
value_rows=[]
for col,p in enumerate(ORDER):
    ax=axes[0,col]; bax=axes[1,col]
    cols=sorted([c for c in S if c.startswith(f"z_cf_{p}_")],key=lambda x:int(x.rsplit("_",1)[1]))
    d=S[S.part==p].dropna(subset=cols); donor=d.var_donor.astype(int).to_numpy(); pred=d[cols].to_numpy().argmax(1)
    M=np.zeros((len(cols),len(cols)))
    for a,b in zip(donor,pred):
        if 0<=a<len(cols): M[a,b]+=1
    M=M/np.maximum(M.sum(1,keepdims=True),1); diag[p]=(donor==pred).mean()
    im=ax.imshow(M,vmin=0,vmax=1,cmap="magma"); ax.set_title(f"{p}\ndiagonal={diag[p]:.2f}")
    ax.set_xticks(np.arange(len(cols))); ax.set_yticks(np.arange(len(cols)))
    ax.set_xlabel("highest-scoring value"); ax.set_ylabel("inserted value")
    groups=[]; labels=[]
    for v,g in d.groupby("var_donor"):
        groups.append(g.margin.to_numpy()); labels.append(str(int(v)))
        value_rows.append({"part":p,"donor_value":int(v),"n":len(g),"median_margin":g.margin.median(),
            "q25_margin":g.margin.quantile(.25),"q75_margin":g.margin.quantile(.75),
            "event_rate":g.responded_but_source_wins.mean()})
    bax.boxplot(groups,tick_labels=labels,showfliers=False,whis=(5,95)); bax.axhline(0,color="black",lw=.8)
    upper=max(float(np.nanmax(np.concatenate(groups))),float(bax.get_ylim()[1]))
    for xpos,g in enumerate(groups,start=1):
        bax.text(xpos,upper,f"n={len(g)}",ha="center",va="bottom",fontsize=6,rotation=90)
    bax.set_ylim(top=upper+max(1,.08*abs(upper)))
    bax.set_xlabel("inserted value"); bax.set_ylabel("final margin"); bax.set_title(f"{p}: value-wise margins")
colorbar=fig.colorbar(im,ax=list(axes[0]),fraction=.015)
colorbar.set_label("fraction within inserted-value row")
fig.suptitle("Figure 7 · Exact-value attribution and final-margin distributions")
plt.show(); display(pd.Series(diag,name="diagonal_rate").to_frame().round(3)); display(pd.DataFrame(value_rows).round(3))


- **Method in one line:** For each swapped part, we took the highest post-swap raw logit within that part block, cross-tabulated it against the inserted exact value, normalized each inserted-value row, and retained every swap.


    ### Plain-language reference for Figure 7

    **Plain caption.** Exact inserted-value recognition is weakest for tail and graded across the other parts, showing that value-level visual difficulty accompanies the swap failures.

    **Terms and how to read it.** A heatmap row is the inserted exact value and a column is the highest-scoring value. Row-normalized color is the fraction of that inserted-value population. The lower boxes show final margins and print swap counts.

    **Literal values.** Post-swap donor-value recognition is graded: diagonal rates are tail 0.395, wing 0.977, beak 0.780, foot 0.965, and eye 0.900. Tail value 7 has only 35 swaps, a median final margin of -4.796, and event rate 0.800; beak value 2 is the next conspicuous difficult value at rate 0.363.

    **Interpretation.** After a tail is inserted, the model names the inserted tail value as its top tail answer only 39.5% of the time. It is much better for wing, foot, and eye. This establishes model-level difficulty choosing among tail's nine exact alternatives; it does not by itself establish that tails are visually ambiguous to a person. Visibility, label conflict, the number of alternatives, and source context remain separate candidate contributors.

**Strongest alternative explanation.** Different parts have different numbers and frequencies of variants, so raw diagonal rates are not directly interchangeable.

    **Discriminating test.** Relate each donor value to species support and its part's alternative count.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR exact-value difficulty as an additional graded contributor across all five parts, not as a tail-only explanation.`

    **Next question.** Do rarity or a larger choice set organize those value-level failures?


## 7b · Are difficult values simply rare or drawn from a larger alternative set?

**Question.** Are difficult values simply rare or drawn from a larger alternative set?

**Variables and prediction.** For every exact donor value, compare its species support with all three mutually exclusive outcomes from Figure 4b; also report the total number of alternatives for its part. If rarity organizes the result, lower-support values should systematically win less or fail more. A mixed pattern rejects support as a sufficient explanation.

**Method.** Label every exact value, print its swap-row denominator, and verify that its three outcome fractions sum to one.

### Figure 7b · Are difficult values simply rare or drawn from a larger alternative set?

**How to read the figure.** Each labelled point is one exact donor value. The x-axis is its species
support: the number of the 50 FunnyBird species that naturally carry that
value in an unmodified bird. It is not an image count or swap count. The
count comes from the renderer's species-to-part-value definition: if six
species ordinarily have donor value 2, its support is 6 even when the swap
table contains hundreds of value-2 rows. The
three panels partition all swaps into donor wins, donorward movement that
remains source-negative, and no donorward movement while source-negative.
The three fractions sum to one within each value. A consistent relationship
with support would make rarity a plausible organizer. The number of
alternatives is reported but cannot be cleanly separated with only five
parts.


In [ ]:
# ALT: Three compact labelled FunnyBird exact-value panels showing all parts together: species support against donor wins, donorward-but-source-still-wins events, and no-donorward-movement failures.
VALUE_OUTCOMES=S.assign(
    donor_wins=S.m_cf>0,
    helped_but_source_wins=(S.m_cf<=0)&(S.response_delta>0),
    no_donorward_move_and_source_wins=(S.m_cf<=0)&(S.response_delta<=0),
)
VS=(VALUE_OUTCOMES.groupby(["part","var_donor"]).agg(
     n_rows=("margin","size"),species_support=("sid_donor","nunique"),
     donor_wins_rate=("donor_wins","mean"),
     responded_but_source_wins_rate=("helped_but_source_wins","mean"),
     no_donorward_move_rate=("no_donorward_move_and_source_wins","mean"),
     median_margin=("margin","median")).reset_index())
VS["alternatives_in_part"]=VS.part.map({p:hi-lo for p,(lo,hi) in SPANS.items()})
sums=VS[["donor_wins_rate","responded_but_source_wins_rate","no_donorward_move_rate"]].sum(axis=1)
if not np.allclose(sums,1): raise RuntimeError("value-level outcome fractions do not sum to one")
panels=[("donor_wins_rate","A · Donor finishes higher"),
        ("responded_but_source_wins_rate","B · Donorward, but source stays higher"),
        ("no_donorward_move_rate","C · No donorward move; source stays higher")]
fig,axes=plt.subplots(1,3,figsize=(18,6),sharex=True,sharey=True)
for ax,(column,title) in zip(axes,panels):
    for part in ORDER:
        d=VS.query("part == @part").sort_values("var_donor")
        ax.scatter(d.species_support,d[column],s=52,color=COLORS[part],label=part)
        for k,r in enumerate(d.itertuples()):
            vertical=5 if k%2==0 else -14
            ax.annotate(f"{part[0]}{int(r.var_donor)}",
                        (r.species_support,getattr(r,column)),fontsize=7,
                        xytext=(4,vertical),textcoords="offset points")
    ax.set_title(title,fontsize=10)
    ax.set_xlabel("species support\n(number of 50 species)")
    ax.set_ylim(-.04,1.04); ax.set_xlim(0,22); ax.grid(alpha=.18)
axes[0].set_ylabel("fraction of swaps for that exact donor value")
axes[2].legend(title="part",fontsize=8,loc="upper right")
fig.suptitle("Figure 7b · Exact-value support versus all three swap outcomes")
plt.tight_layout(); plt.show(); display(VS.round(3))


- **Method in one line:** We counted how many of the 50 species naturally carry each donor value, then grouped all swap rows for that value into the three exhaustive Figure 4b outcomes; no correlation model was fitted.


    ### Plain-language reference for Figure 7b

    **Plain caption.** Rarity sometimes accompanies difficult exact values, but support does not cleanly determine whether the donor wins, helps but loses, or fails to move the score.

    **Terms and how to read it.** Species support is the number of the 50 species naturally carrying an exact value. The three y-axes are mutually exclusive fractions with all swaps for that donor value as denominator; together they sum to one.

    **Literal values.** All three outcome panels use every swap for each donor value and close to one. Lower-support values are sometimes difficult—tail value 7 has support from two species and controlled-event rate 0.800, and beak value 2 has support from six species and rate 0.363—but donor wins, insufficient donorward corrections, and no-movement failures are not monotone in support within every part. Tail remains difficult and wing/foot strong across overlapping support values.

    **Interpretation.** The three clouds distinguish three possible rarity stories. A rare value can win less often, be noticed but fail to overcome the source, or fail to move donorward at all. Upper-left points support a rarity concern; upper-right and lower-left counterexamples show that support is not a complete rule. Tail remains worse than wing and foot at overlapping support, so another part-specific contributor is required.

**Strongest alternative explanation.** The number of alternatives is constant within a part and therefore remains confounded with all other part-level differences.

    **Discriminating test.** Use more independent part families or a design that changes choice-set size while holding pixels and species fixed.

    **Verdict.** **KEEP**.

    **Proof ledger.** `VALID TEST, NO CLEAR SUPPORT that frequency or alternative count alone explains the part ordering. The all-outcome display is complete.`

    **Next question.** Does unchanged source species organize what remains after exact values?


## 8 · Does source species organize the remaining error after exact values?

**Question.** If two birds receive the same exact replacement, do their
source species still accompany systematically different final margins?

**Why exact-pair centering is needed.** A species could appear resistant
merely because it happens to receive difficult replacements. We first
compare each row only with swaps having the same part, old exact value,
and inserted exact value. This removes that composition difference before
species are summarized.

**Complete procedure.**

```text
for each swap row:
    exact_pair = (part, source_value, donor_value)

for each exact_pair:
    pair_mean = average final margin across all its rows

for each row:
    row_residual = row final margin - its pair_mean

for each (part, source_species):
    species_residual = average row_residual

display species only when it has at least five rows
```

**Numerical example.** Suppose red-tail to blue-tail margins are `-5`
and `-3` for Species A and `+1` and `+3` for Species B. Their exact-pair
mean is `-1`. The residuals are therefore `-4,-2,+2,+4`: they average to
zero over the pair, but Species A averages `-3` and Species B averages
`+3`. Species A is more source-retaining than the exact-pair average.

**Prediction and limit.** Persistent species residuals support an
unchanged-body/species association after exact values. They do not prove
that species causes the difference because species remains bundled with
body shape, pose tendencies, visibility, and other renderer properties.

### Figure 8 · Source-species residual after exact source/donor values

**How to read the figure.** Every row keeps one source-species identity
and every column keeps one replaced part. Blue cells are more
source-retaining than other swaps with the same exact source and donor
values; red cells are more donor-receptive; white is the exact-pair
average. Blank cells lack five eligible rows. Reading across one row asks
whether that unchanged bird context accompanies similar or different
residuals for several parts; no correlation threshold is imposed.


In [ ]:
# ALT: Common source-species-by-part heatmap of final-margin residuals after exact-pair centering, preserving every displayed species identity.
R=S.copy(); R["value_pair_mean"]=R.groupby(["part","var_src","var_donor"]).margin.transform("mean")
R["margin_after_value_pair"]=R.margin-R.value_pair_mean
pair_zero=R.groupby(["part","var_src","var_donor"]).margin_after_value_pair.mean().abs().max()
if pair_zero>1e-10: raise RuntimeError(f"exact-pair residual means do not close: {pair_zero}")
SP=(R.groupby(["part","sid_src"]).agg(n=("margin","size"),residual=("margin_after_value_pair","mean"))
      .reset_index().query("n>=5"))
species_matrix=SP.pivot(index="sid_src",columns="part",values="residual").reindex(columns=ORDER)
species_spread=(SP.groupby("part").residual.agg(["min","median","max","std","count"])
                .reindex(ORDER))
species_matrix=species_matrix.loc[species_matrix.mean(axis=1,skipna=True).sort_values().index]
lim=float(np.nanmax(np.abs(species_matrix.to_numpy())))
fig,ax=plt.subplots(figsize=(10,max(8,.24*len(species_matrix))))
im=ax.imshow(species_matrix.to_numpy(),aspect="auto",cmap="coolwarm",vmin=-lim,vmax=lim)
ax.set_xticks(np.arange(len(ORDER)),ORDER)
ax.set_yticks(np.arange(len(species_matrix)),
              [f"species {int(x)}" for x in species_matrix.index],fontsize=7)
ax.set_xlabel("replaced part"); ax.set_ylabel("unchanged source species")
ax.set_title("Figure 8 · Source-species residual after exact source/donor values")
fig.colorbar(im,ax=ax,fraction=.035,pad=.02).set_label(
    "mean final-margin residual (blue=source-retaining, red=donor-receptive)")
plt.tight_layout(); plt.show()
display(species_spread.round(3))
print("maximum absolute within-exact-pair residual mean:",pair_zero)


- **Method in one line:** We subtracted each ordered `(part, source value, donor value)` pair's pooled mean margin and averaged the remaining residuals by unchanged source species; this is descriptive centering, not a fitted causal model.


    ### Plain-language reference for Figure 8

    **Plain caption.** After matching the exact source and donor values, source species still organize final margins descriptively, but the plot does not isolate a causal species effect.

    **Terms and how to read it.** An exact pair is `(part, source value, donor value)`. A row residual is its final margin minus that pair's pooled mean. Species residual is the average of those row residuals for one source species; within-pair residuals average to zero, but within-species residuals need not.

    **Literal values.** After centering each part/source-value/donor-value combination, source-species mean residuals remain nonzero in every part. Their standard deviations are tail 2.043, beak 1.733, eye 1.466, foot 1.342, and wing 1.341 raw-logit units; tail ranges from -4.173 to 8.259.

    **Interpretation.** Even for the same source and donor values, some source species shift the margin upward and others downward. This says the unchanged bird background is associated with the answer, but this particular plot reuses the same rows to estimate and summarize the shift, so it is descriptive.

**Strongest alternative explanation.** The descriptive species means can also absorb pose or repeated-row composition, and they are not a causal body manipulation.

    **Discriminating test.** Check whether species is recoverable from held-out concept vectors and whether species improves held-out margin prediction.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR observational source-species variation beyond exact values. The common heatmap preserves identity across parts, but neither repeated nor part-specific color patterns establish a causal species effect.`

    **Next question.** Is species information actually present in the learned concept representation?


## 8b · How much species identity is recoverable from the learned concept vector?

**Question.** How much species identity is recoverable from the learned concept vector?

**Variables and prediction.** After the CBM is finished, train three read-only diagnostic classifiers. Repeat the test once with the complete five-part recipe and once with each part alone. Grey receives official yes/no answers c; solid color receives model raw scores z; outline receives each raw score after subtracting the training-fold average for the same yes/no answer. Raw or residual accuracy above the known-label control means score magnitudes reveal species beyond the nominal concept pattern. It does not say which pixels produced them, whether the saved class head uses them, or whether they caused a swap failure.

**Method.** Use one fixed stratified 70/30 split of the held-out prediction population.

### Figure 8b · How much species identity is recoverable from the learned concept vector?

**How to read the figure.** The y-axis is held-out species-classification accuracy. For every block, grey
uses processed 0/1 concept labels, color uses learned raw logits, and the
outlined bar uses raw logits after the training-fold mean for the same 0/1
label has been removed. The grey bar is the structural control: with balanced
FunnyBird species and `K` mutually exclusive values for one part, it is
approximately `K/50` (tail has 9 values, so 9/50=0.18), not 1/50. The residual
bar asks whether score magnitudes still identify species after the nominal
label bucket is removed. This diagnoses available information only; it does
not prove that the saved CBM uses it or that it caused backwash.


### Before Figure 8b: what exactly are the grey and colored bars?

Each image has 26 official yes/no answers, written `c`. The word
**binary** means only that an answer is 0 for “absent” or 1 for “present.”
For example, a bird can have `beak_0=1` and the other beak values equal
to 0. Across the five parts, these 26 answers are simply a long way to
record the bird's complete tail + wing + beak + foot + eye recipe. They
are dataset facts, not model scores.

**Why can all official answers identify 100% of species while one part
cannot?** The 50 synthetic species have distinct *combinations* of the
five part values. Several species can share the same tail, so tail alone
leaves several possible species. Adding wing, beak, foot, and eye can
make the complete combination unique. It is like a five-digit code:
one digit is shared by many records, while all five digits together can
identify one record. Therefore the 100% bar for all five parts is
expected dataset structure; it does not mean any single concept head is
making a 50-species prediction.

We train separate diagnostic classifiers whose target is the species `y`:

- **grey bar:** input is the corresponding official yes/no answers `c`;
- **solid colored bar:** input is the corresponding learned raw scores `z`;
- **outlined bar:** input is `z` after subtracting the training-fold mean
  for the same exact concept and 0/1 label. This asks whether magnitudes
  still identify species *within* the official label buckets;
- **bar height:** held-out species accuracy of that diagnostic classifier.

Thus “species information is present” means only that a classifier can guess
species from the supplied numbers better than chance. It does **not** mean the
saved CBM classified the image with that accuracy, and it does **not** measure
whether a concept used its named pixels.

Example: suppose a purple tail is shared by species 4, 12, 19, 31, and
44. The official tail answer can narrow the choice to those five species
but cannot say which of the five is present. If the nine raw tail scores
nevertheless differ systematically among those species, a diagnostic can
do better than the official tail answers. That extra performance is the
within-bucket species information being tested.

A single part block is supplied as several numbers to a multinomial
logistic regression. For example, the nine tail scores become nine input
columns, and the diagnostic fits 50 weighted sums—one per species. We are
not zeroing other weights in the saved CBM because this is a new diagnostic
classifier. Figure 8c instead asks the swap-specific question that this
ordinary-image probe cannot answer: whether post-swap scores retain the
unchanged source species after exact source and donor values are controlled.

The held-out probe population is only 30% of 500 images: 150 images, or
roughly three per species. Therefore small differences between part bars
can correspond to only a few images. Use this test to establish that
information is available, not to claim a precise causal ranking.

> **IMPORTANT: Species leakage makes backwash possible, but leakage alone does
> not cause it. Wing is the clearest counterexample: wing `z` reveals species,
> yet the controlled swaps show strong grounding.**

What predicts grounding is measured separately: `response_delta`, final margin
`m_cf`, target-part visibility, label/mask conflict, and exact donor-value
recognition. Figure 8b is an availability/control diagnostic, not that outcome.


In [ ]:
# ALT: One horizontal comparison showing held-out species decoding from the complete five-part recipe versus each part alone, using official answers, raw scores, and within-answer raw-score remainders.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
idx=np.arange(len(y_saved)); tr,te=train_test_split(idx,test_size=.30,random_state=20260803,stratify=y_saved)
blocks={"all five parts together\n(26 outputs)":np.arange(z_saved.shape[1])}
blocks.update({p:np.arange(lo,hi) for p,(lo,hi) in SPANS.items()})
label_means=np.zeros((z_saved.shape[1],2),dtype=float)
for j in range(z_saved.shape[1]):
    for label in [0,1]:
        rows=tr[c_saved[tr,j].astype(int)==label]
        if not len(rows): raise RuntimeError(f"no training-fold rows for concept {j}, label {label}")
        label_means[j,label]=z_saved[rows,j].mean()
residual_z=z_saved.copy()
for j in range(z_saved.shape[1]):
    residual_z[:,j]-=label_means[j,c_saved[:,j].astype(int)]
probe=[]
for name,cols in blocks.items():
    raw_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    label_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    residual_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    raw_model.fit(z_saved[tr][:,cols],y_saved[tr]); label_model.fit(c_saved[tr][:,cols],y_saved[tr])
    residual_model.fit(residual_z[tr][:,cols],y_saved[tr])
    probe.append({"block":name,
                  "raw_z_accuracy":accuracy_score(y_saved[te],raw_model.predict(z_saved[te][:,cols])),
                  "within_label_residual_accuracy":accuracy_score(y_saved[te],residual_model.predict(residual_z[te][:,cols])),
                  "processed_label_accuracy":accuracy_score(y_saved[te],label_model.predict(c_saved[te][:,cols])),
                  "label_patterns":len(np.unique(c_saved[tr][:,cols],axis=0)),"dimensions":len(cols)})
PROBE=pd.DataFrame(probe)
plot_order=["all five parts together\n(26 outputs)"]+ORDER
P=PROBE.set_index("block").reindex(plot_order).reset_index()
y=np.arange(len(P)); h=.23
block_colors=["#333333"]+[COLORS[p] for p in ORDER]
fig,ax=plt.subplots(figsize=(12,7))
ax.barh(y-h,100*P.processed_label_accuracy,h,label="official yes/no answers",color="#BBBBBB")
ax.barh(y,100*P.raw_z_accuracy,h,label="model's raw scores",color=block_colors)
ax.barh(y+h,100*P.within_label_residual_accuracy,h,
        label="raw-score remainder within the same yes/no answer",
        facecolor="white",edgecolor=block_colors,hatch="//")
ax.set_yticks(y,P.block); ax.invert_yaxis(); ax.set_xlim(0,105)
ax.axvline(2,color="#666666",ls=":",lw=1,label="50-species chance (2%)")
ax.set_xlabel("species identified correctly among 150 held-out images (%)")
ax.set_title("Figure 8b · All five parts together versus one shared part alone")
ax.legend(fontsize=9,loc="lower right")
for yy,row in P.iterrows():
    for offset,value in [(-h,row.processed_label_accuracy),(0,row.raw_z_accuracy),(h,row.within_label_residual_accuracy)]:
        ax.text(100*value+1,yy+offset,f"{100*value:.1f}%",va="center",fontsize=8)
ax.text(52,-.58,"Complete five-part recipe",ha="center",va="center",fontsize=10,fontweight="bold")
ax.axhline(.5,color="#777777",lw=.8)
ax.text(102,3.0,"Each row below\nuses one part only",ha="right",va="center",fontsize=9)
plt.tight_layout(); plt.show(); display(P.round(3))


- **Method in one line:** We fitted three new multinomial logistic-regression diagnostic classifiers per concept block on 70% of the frozen CBM's held-out images and tested them on the same remaining 30%: one used binary labels, one raw logits, and one within-label residual logits; the CBM itself was not retrained or altered.


    ### Plain-language reference for Figure 8b

    **Plain caption.** The complete five-part recipe identifies species, while one part alone leaves several possible species. Raw concept-score magnitudes distinguish some species inside those shared-part groups; this is information availability, not yet use or grounding failure.

    **Terms and how to read it.** Each row says whether the probe receives all five parts together or only one named part. Grey uses official yes/no answers; solid color uses model raw scores; outline uses the raw-score remainder after subtracting the average for the same yes/no answer. The x-axis is the percentage of 150 held-out images whose species is identified correctly.

    **Literal values.** On the same held-out split, the complete five-part recipe identifies species at 1.000 from official labels and 0.993 from all 26 raw logits. This does not mean one part identifies every species: individual raw-z blocks greatly exceed their label controls: beak 0.407 versus 0.080, eye 0.233 versus 0.060, foot 0.347 versus 0.080, tail 0.953 versus 0.180, and wing 0.700 versus 0.120. Even after the training-fold mean for each 0/1 label bucket is removed, held-out accuracy remains 0.260 for beak, 0.127 for eye, 0.180 for foot, 0.727 for tail, 0.333 for wing, and 0.947 for all 26 scores.

    **Interpretation.** The complete official five-part recipe already identifies the synthetic species, but one shared part alone does not. Within each one-part test, the model's score magnitudes reveal more species identity than that part's official answers. For example, tail scores identify species 95.3% of the time although the nine tail answers alone reach only 18.0%. This extra information is present, but presence alone does not prove it caused a replacement failure. Wing is the decisive control: its raw scores decode species well, yet its exact donor recognition and donorward movement are strong enough that backwash is rare. The emerging theory is competition between retained source-associated structure and local donor evidence.

**Strongest alternative explanation.** Species decodability is not grounding: a score block can identify species while still responding correctly to its named pixels, as the controlled wing swaps show.

    **Discriminating test.** Judge grounding from response_delta and the final donor-minus-source margin, then relate those outcomes to visibility, conflict, and exact-value recognition.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR species-information availability beyond the official 0/1 label buckets. This is not a grounding test and does not show that leakage alone causes backwash.`

    **Next question.** After a controlled replacement, do the replaced-part scores retain the unchanged source species?


In [ ]:
# ALT: Numbered comparison table placing species decoding beside donorward movement, exact-value recognition, and controlled backwash for every part.
grounding_comparison=[]
probe_by_block=PROBE.set_index("block")
for part in ORDER:
    d=S[S.part==part]
    grounding_comparison.append({
        "part":part,
        "species_decoding_from_raw_scores":probe_by_block.loc[part,"raw_z_accuracy"],
        "mean_donorward_movement":d.response_delta.mean(),
        "inserted_value_recognition":diag[part],
        "controlled_backwash_rate":d.responded_but_source_wins.mean(),
    })
GROUNDING_COMPARISON=pd.DataFrame(grounding_comparison)
display(Markdown("""
### Comparison table 8b.1 · Why species decoding is not backwash

The four columns answer different questions. Species decoding asks what
information a new diagnostic can recover. Donorward movement asks how far
the named concept comparison changes after the pixels change. Recognition
asks whether the exact inserted value becomes the largest value in its
part. Controlled backwash requires a positive movement but a final source
win. If species decoding alone caused backwash, these columns would follow
the same part ordering.
"""))
display(GROUNDING_COMPARISON.round(3))
display(Markdown("""
**Literal comparison.** Wing is the decisive counterexample: its scores
decode species well, but wing has the largest donorward movement, nearly
perfect inserted-value recognition, and very little controlled backwash.
Therefore species information is available but is not sufficient to
produce the failure.
"""))


- **Method in one line:** We placed already-computed part summaries side by side without fitting a model, specifically to test whether species decoding, donorward response, exact-value recognition, and controlled backwash share the same part ordering.


## 8c · After replacement, does the part block retain the unchanged source species?

**Why this test is needed.** Figure 8b uses ordinary images. It shows that
species information is available in raw concept scores, but wing proves
that availability alone is not backwash. The mechanism hypothesis is more
specific: after the named part changes, failed part-score blocks may still
retain more information about the unchanged source bird than successful
blocks do.

**Variables and controls.** For each part, the inputs are all post-swap
raw logits for that part: nine tail scores, six wing, four beak, four foot,
or three eye. Inside every training fold, subtract the mean score vector
for the same exact `(source value, donor value)` pair. This prevents the
diagnostic from succeeding merely because particular species receive
particular value pairs. Every swap from one original source image remains
in one fold.

Two read-only predictions are compared:

1. **exact-pair baseline:** predict source species from the exact source
   and donor values alone;
2. **post-swap residual logits:** predict source species from the remaining
   part-score pattern after the exact-pair mean is removed.

Accuracy is reported separately for donor wins, donorward movement that
remains source-negative, and no-donorward-movement failures. These are
repeated swap rows from one seed, so they are diagnostic denominators—not
seed-level uncertainty. An outcome bar is interpreted only when it contains
at least 25 distinct original source images. Smaller groups remain in the
printed audit table but are marked as insufficient rather than plotted as
reliable accuracies. For example, three swap rows from one original image
cannot support a 50-species decoding claim even if their observed accuracy
happens to be 100%.

**Prediction.** Retained source context supports the proposed mechanism
only if residual post-swap scores identify the unchanged source beyond the
exact-pair baseline and do so more strongly for controlled failures than
donor wins. Tail should show a clearer contrast than wing. Equal decoding
across outcomes would mean species information is available but does not
discriminate the backwash event.

### Figure 8c · Held-out unchanged-source information in post-swap part scores


In [ ]:
# ALT: Five part panels comparing exact-pair baseline and residual post-swap-logit accuracy for predicting unchanged source species across the three controlled outcomes.
from sklearn.model_selection import StratifiedKFold
source_id_column=next((c for c in ["orig_render_id","source_render_id","li","image_orig","orig_image"] if c in S),None)
if source_id_column is None:
    raise RuntimeError("Figure 8c requires original source-image identity")
source_units=(S[[source_id_column,"sid_src"]].drop_duplicates()
                .sort_values(source_id_column).reset_index(drop=True))
if source_units.groupby(source_id_column).sid_src.nunique().max()!=1:
    raise RuntimeError("one original image maps to multiple source species")
splitter=StratifiedKFold(n_splits=5,shuffle=True,random_state=20260901)
unit_fold={}
for fold,(_,test_idx) in enumerate(splitter.split(source_units[source_id_column],source_units.sid_src)):
    for unit in source_units.iloc[test_idx][source_id_column].astype(str): unit_fold[unit]=fold

retention_predictions=[]
for part in ORDER:
    d=S[S.part==part].copy()
    feature_columns=sorted([c for c in d if c.startswith(f"z_cf_{part}_")],
                           key=lambda x:int(x.rsplit("_",1)[1]))
    if len(feature_columns)!=(SPANS[part][1]-SPANS[part][0]):
        raise RuntimeError(f"Figure 8c missing post-swap {part} block")
    d["fold"]=d[source_id_column].astype(str).map(unit_fold)
    d["exact_pair"]=d.var_src.astype(int).astype(str)+"->"+d.var_donor.astype(int).astype(str)
    d["outcome"]=np.select(
        [d.m_cf>0,(d.m_cf<=0)&(d.response_delta>0)],
        ["donor wins","donorward, source wins"],
        default="no donorward move")
    for fold in range(5):
        tr=d[d.fold!=fold].copy(); te=d[d.fold==fold].copy()
        global_mean=tr[feature_columns].mean()
        pair_means=tr.groupby("exact_pair")[feature_columns].mean()
        def residualize(frame):
            means=frame[["exact_pair"]].join(pair_means,on="exact_pair")[feature_columns]
            means=means.fillna(global_mean)
            return frame[feature_columns].to_numpy()-means.to_numpy()
        xtr=residualize(tr); xte=residualize(te)
        diagnostic=make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=4000,C=1.0,random_state=20260901))
        diagnostic.fit(xtr,tr.sid_src.astype(int))
        raw_prediction=diagnostic.predict(xte)
        pair_lookup=(tr.groupby("exact_pair").sid_src
                       .agg(lambda x:int(x.value_counts().index[0])))
        global_species=int(tr.sid_src.value_counts().index[0])
        pair_prediction=te.exact_pair.map(pair_lookup).fillna(global_species).astype(int).to_numpy()
        for row_index,raw_pred,pair_pred in zip(te.index,raw_prediction,pair_prediction):
            retention_predictions.append({
                "row_index":row_index,"part":part,"fold":fold,
                "outcome":d.loc[row_index,"outcome"],
                "source_species":int(d.loc[row_index,"sid_src"]),
                "original_image":str(d.loc[row_index,source_id_column]),
                "raw_residual_correct":int(raw_pred==int(d.loc[row_index,"sid_src"])),
                "exact_pair_correct":int(pair_pred==int(d.loc[row_index,"sid_src"])),
            })
RETENTION_ROWS=pd.DataFrame(retention_predictions)
RETENTION=(RETENTION_ROWS.groupby(["part","outcome"]).agg(
    n_rows=("source_species","size"),
    n_original_images=("original_image","nunique"),
    exact_pair_accuracy=("exact_pair_correct","mean"),
    residual_logit_accuracy=("raw_residual_correct","mean")).reset_index())
MIN_OUTCOME_IMAGES=25
RETENTION["coverage_ok"]=RETENTION.n_original_images>=MIN_OUTCOME_IMAGES
outcome_order=["donor wins","donorward, source wins","no donorward move"]
fig,axes=plt.subplots(1,5,figsize=(21,5.2),sharey=True)
x=np.arange(len(outcome_order)); width=.34
for ax,part in zip(axes,ORDER):
    p=(RETENTION[RETENTION.part==part].set_index("outcome")
         .reindex(outcome_order))
    shown_pair=p.exact_pair_accuracy.where(p.coverage_ok)
    shown_residual=p.residual_logit_accuracy.where(p.coverage_ok)
    ax.bar(x-width/2,shown_pair,width,color="#BBBBBB",label="exact-pair baseline")
    ax.bar(x+width/2,shown_residual,width,color=COLORS[part],label="residual post-swap logits")
    ax.axhline(1/50,color="#666666",ls=":",lw=1)
    ax.set_xticks(x,["donor\nwins","helps,\nsource wins","no donor\nmove"],fontsize=8)
    ax.set_ylim(0,1); ax.set_title(part)
    for xx,row in enumerate(p.itertuples()):
        if pd.notna(row.n_rows):
            if bool(row.coverage_ok):
                label=f"rows={int(row.n_rows)}\nimages={int(row.n_original_images)}"
            else:
                label=f"insufficient\n{int(row.n_original_images)} images"
            ax.text(xx,.96,label,ha="center",va="top",fontsize=7)
axes[0].set_ylabel("held-out unchanged-source species accuracy")
axes[-1].legend(fontsize=8,loc="upper right")
fig.suptitle("Figure 8c · Does the replaced-part score block retain the unchanged source species?")
plt.tight_layout(); plt.show(); display(RETENTION.round(3))


- **Method in one line:** We used five source-image-grouped folds, removed each training fold's exact source/donor-pair mean from the post-swap part block, fitted a new read-only species diagnostic on four folds, and compared its held-out accuracy with an exact-pair-only baseline; the CBM and swaps were unchanged.


In [ ]:
# ALT: Executed plain-language review of Figure 8c with the predeclared tail-versus-wing and controlled-failure-versus-donor-win decision.
interpretable=RETENTION[RETENTION.coverage_ok].copy()
pivot=interpretable.pivot(index="part",columns="outcome",values="residual_logit_accuracy").reindex(ORDER)
contrast=(pivot.get("donorward, source wins")-pivot.get("donor wins")).rename("failure_minus_donor_win")
tail_contrast=float(contrast.loc["tail"])
wing_contrast=float(contrast.loc["wing"]) if pd.notna(contrast.loc["wing"]) else np.nan
excluded=(RETENTION.loc[~RETENTION.coverage_ok,["part","outcome","n_original_images"]]
          .sort_values(["part","outcome"]).to_dict("records"))
if tail_contrast>0 and (not np.isfinite(wing_contrast) or tail_contrast>wing_contrast):
    interpretation=("Tail retains more held-out source identity in controlled failures than in donor wins, "
                    "and its contrast exceeds wing. This supports retained source context as an observational "
                    "mechanism candidate, while the exact-pair control prevents a simple value-composition explanation.")
    verdict="KEEP as focused observational support; causal isolation still belongs to a context intervention"
else:
    interpretation=("Residual post-swap source decoding does not show the predicted tail-specific failure contrast. "
                    "Species information remains available, but this test does not promote it to an explanation of controlled backwash.")
    verdict="KEEP as a valid discriminating negative result; do not claim source retention explains backwash"
display(Markdown(f"""
### Plain-language reference for Figure 8c

**Plain caption.** This test asks whether the replaced part's score pattern
still identifies the unchanged source bird after the exact old and inserted
values have been accounted for, and whether that retained identity is
concentrated in swaps where the source answer survives.

**Terms and denominators.** Grey is the held-out exact-pair-only baseline.
Color is a newly fitted read-only classifier using the residual post-swap
logits. Each `n` is a swap-row denominator; `n_original_images` in the
printed table shows the underlying source-image coverage. The dotted 0.02
line is blind 50-species chance. Bars require at least
`{MIN_OUTCOME_IMAGES}` distinct original source images; smaller groups are
labelled “insufficient” and remain only in the audit table. No bar is a
seed-level error estimate.

**Literal result.** The complete executed table above gives baseline and
residual-logit accuracy for every part and outcome. The controlled-failure
minus donor-win residual-decoding contrasts are:

`{contrast.dropna().round(3).to_dict()}`.

Outcome groups excluded from comparison for inadequate original-image
coverage are: `{excluded}`.

**Interpretation.** {interpretation}

**Alternative.** Residual logits can retain pose, visibility, or renderer
regularities bundled with species. A diagnostic classifier establishes
recoverable information, not an independently manipulated cause.

**Discriminating test.** Compare the matched Standard and RLv2 fixed swaps,
and then test whether MCBM gamma compresses the same information while
changing the controlled outcome.

**Verdict.** **{verdict}.**

**Proof ledger.** This is the sole swap-specific source-retention diagnostic;
it does not replace the renderer predicate in Figure 4.

**Next question.** Which proposed contributor blocks improve prediction
for entirely held-out original images, and what residual remains?
"""))


## 9 · How accurately can progressively richer grouping information predict unseen margins?

**Question.** Do visibility, exact values, and source species predict the
final raw-logit margin for original source images that were not used to
build the prediction rule?

**What a prediction rule is.** It is a lookup learned from training folds,
not another neural network. For example, the rule can learn that visible
tail swaps in its training rows have mean margin `-2`, then predict `-2`
for a held-out visible tail swap. The prediction is compared with the
held-out observed margin.

**Five-fold procedure.**

```text
assign each original source image to one of five folds
keep every swap from that image in the same fold

for each held-out fold:
    use the other four folds to estimate group means
    blend each group mean with 10 virtual rows at the overall mean
    predict the untouched fold

combine predictions from all five held-out folds
calculate RMSE and MAE
```

The ten virtual rows shrink tiny groups toward the overall mean so one or
two unusual rows cannot create an extreme lookup. They are a declared
regularization choice, not additional observations.

**What the x-axis means.** “Part only” gives one learned mean per part.
“+ visibility” learns separate means for the declared pixel-count bins.
“+ exact values” additionally separates old and inserted values. “+ source
species” additionally separates the unchanged source species. Each stage
contains every field from the previous stage.

**Outcome and prediction.** RMSE is the square root of the mean squared
difference between predicted and observed final margins, in raw-logit
units. Lower is better. A change from 4 to 3 means a typical held-out
error reduction of roughly one logit unit; 3 to 4 is worsening and gives
that added block no explanatory credit. The remaining error is a measured
residual, not automatically an unknown causal mechanism.

### Figure 9 · Held-out margin prediction using progressively richer grouping information

**How to read the figure.** Panel A gives the four held-out errors. Large
points combine all folds; small translucent points diagnose whether the
direction is confined to one fold and are not seed-level error bars.
Lower is better. Panel B gives exact-group coverage: the percentage of
held-out rows whose full grouping key was observed in the training folds.
Falling coverage and small training groups demonstrate when a richer rule
becomes sparse rather than merely asserting that explanation afterward.
The tables print RMSE, MAE, coverage, group counts, split unit, and all
fold-level diagnostics. Figure 6 already answers the separate descriptive
visibility-selection question.


In [ ]:
# ALT: Held-out final-margin prediction error when a post-hoc rule receives progressively richer FunnyBird grouping information.
import hashlib
A=S.copy(); A["vis_bin"]=pd.cut(A.pixel_count_cf,[-1,19,49,99,199,499,np.inf],labels=False)
original_id_column=next((c for c in ["orig_render_id","source_render_id","li","image_orig","orig_image"] if c in A),None)
if original_id_column is None:
    raise RuntimeError("Figure 9 requires an original source-image identity; swap-row render_id is not an independent split unit")
unit=A[original_id_column].astype(str)
A["fold"]=unit.map(lambda x:int(hashlib.sha1(x.encode()).hexdigest(),16)%5)
if A.groupby(original_id_column).fold.nunique().max()!=1:
    raise RuntimeError("one original source image was assigned to more than one fold")
stages=[("part only",["part"]),("+ visibility",["part","vis_bin"]),
        ("+ exact values",["part","vis_bin","var_src","var_donor"]),
        ("+ source species",["part","vis_bin","var_src","var_donor","sid_src"])]
rows=[]; fold_rows=[]
for stage,cols in stages:
    pred=pd.Series(index=A.index,dtype=float)
    for fold in range(5):
        tr=A[A.fold!=fold]; te=A[A.fold==fold]
        prior=tr.margin.mean(); stats=tr.groupby(cols).margin.agg(["mean","count"]).reset_index()
        stats["estimate"]=(stats["mean"]*stats["count"]+prior*10)/(stats["count"]+10)
        joined=te[cols].merge(stats[cols+["estimate"]],on=cols,how="left")
        matched=joined.estimate.notna().to_numpy()
        fold_prediction=joined.estimate.fillna(prior).to_numpy()
        pred.loc[te.index]=fold_prediction
        fold_rows.append({
            "stage":stage,"fold":fold,"n_test_rows":len(te),
            "heldout_group_coverage":float(matched.mean()),
            "training_groups":len(stats),
            "median_training_group_count":float(stats["count"].median()),
            "rmse":float(np.sqrt(np.mean((te.margin.to_numpy()-fold_prediction)**2))),
            "mae":float(np.mean(np.abs(te.margin.to_numpy()-fold_prediction))),
        })
    rows.append({"stage":stage,"rmse":float(np.sqrt(np.mean((A.margin-pred)**2))),
                 "mae":float(np.mean(np.abs(A.margin-pred)))})
ACCOUNT=pd.DataFrame(rows)
ACCOUNT_FOLDS=pd.DataFrame(fold_rows)
coverage=(ACCOUNT_FOLDS.groupby("stage",sort=False).agg(
    mean_group_coverage=("heldout_group_coverage","mean"),
    min_group_coverage=("heldout_group_coverage","min"),
    median_training_group_count=("median_training_group_count","median"),
    mean_training_groups=("training_groups","mean")).reindex(ACCOUNT.stage).reset_index())
ACCOUNT=ACCOUNT.merge(coverage,on="stage",how="left")
ACCOUNT["split_unit"]=original_id_column
ACCOUNT["n_original_images"]=unit.nunique()
fig,axes=plt.subplots(1,2,figsize=(15,5.5))
ax=axes[0]
ax.plot(ACCOUNT.stage,ACCOUNT.rmse,"o-",color="#0072B2",lw=2)
for fold,d in ACCOUNT_FOLDS.groupby("fold"):
    ax.scatter(d.stage,d.rmse,color="#0072B2",alpha=.28,s=22)
ax.set_ylabel("held-out prediction error, RMSE (raw-logit units)")
ax.tick_params(axis="x",rotation=20)
ax.set_xlabel("information available to the post-hoc prediction rule")
ax.set_title("A · Held-out error (large points=all folds)\nsmall points diagnose individual folds")
for row in ACCOUNT.itertuples():
    ax.annotate(f"{row.rmse:.3f}",(row.stage,row.rmse),xytext=(0,8),textcoords="offset points",ha="center")
ax=axes[1]
ax.plot(ACCOUNT.stage,100*ACCOUNT.mean_group_coverage,"o-",color="#D55E00",lw=2)
ax.set_ylim(0,105); ax.set_ylabel("held-out rows with a matching training group (%)")
ax.tick_params(axis="x",rotation=20); ax.set_xlabel("same sequential information stages")
ax.set_title("B · Exact-group coverage\nlow coverage reveals sparse lookup groups")
for row in ACCOUNT.itertuples():
    ax.annotate(f"{100*row.mean_group_coverage:.1f}%",(row.stage,100*row.mean_group_coverage),
                xytext=(0,8),textcoords="offset points",ha="center")
fig.suptitle("Figure 9 · Can added information predict unseen final margins, and are its groups supported?")
plt.tight_layout(); plt.show(); display(ACCOUNT.round(3))
print("Fold-level diagnostic (folds are not training-seed error bars):")
display(ACCOUNT_FOLDS.round(3))


- **Method in one line:** We fitted a five-fold cross-validated, shrinkage-regularized group-mean lookup—not a neural network—while keeping all swaps from each original image in one fold, then calculated held-out RMSE after adding visibility, exact pair, and source species sequentially.


    ### Plain-language reference for Figure 9

    **Plain caption.** Visibility improves prediction for unseen source images, while this particular categorical lookup gives exact values and source species no held-out explanatory credit.

    **Terms and how to read it.** A post-hoc prediction rule maps known row fields to a predicted final margin. Five folds keep every swap from one original image together. Panel A reports RMSE; lower is better, and small fold points are diagnostics rather than seed uncertainty. Panel B reports exact training-group coverage for held-out rows; lower coverage means the richer lookup is increasingly sparse. Nothing is added to the image or CBM.

    **Literal values.** With all swaps from one original image kept in one fold (250 original images), held-out RMSE improves from 3.333 to 3.098 when visibility is added. It then worsens to 3.472 with exact values and 3.801 with source species; MAE follows the same pattern.

    **Interpretation.** Visibility is the only added block that predicts unseen original images better. Adding exact values or source species makes predictions worse, which means this declared test gives them no generalizing explanatory credit even though earlier descriptive plots show associations.

**Strongest alternative explanation.** There are many exact-value and species combinations but only 250 original images, so some training-fold groups are small. The simple group-average predictor may therefore be a poor model, but that possibility cannot be counted as positive evidence.

    **Discriminating test.** Repeat with independent seeds or predeclare a different predictor before assigning generalizing explanatory credit to exact values or species.

    **Verdict.** **KEEP**.

    **Proof ledger.** `VALID TEST, NO SUPPORT from this predictor that exact values or source species account for held-out margin variance; only visibility gives a small improvement.`

    **Next question.** Is the final concept margin associated with downstream donor-species probability?


## Textbook guide: the measurements are related questions, not interchangeable scores

The aligned figures deliberately put anatomical groups in the same row order,
but the panels do **not** all measure the same thing. FunnyBird has controlled
part replacement; CUB has natural photographs and released masks. We therefore
match the scientific question while naming the weaker CUB approximation.

| Scientific question | FunnyBird measurement | CUB measurement | Same operation? |
|---|---|---|---|
| Are labels present without visible part evidence? | renderer-derived label/visibility conflict | positive label with mapped mask absent | related; CUB masks are noisier |
| Is the concept output usable? | raw-`z` spread, balanced accuracy, positive recall | the same health checks | yes |
| Do named pixels affect the score? | controlled `response_delta` after donor insertion | visible-minus-hidden raw-`z` difference | no; CUB compares different photographs |
| Does context remain after local evidence is limited? | donorward response occurs but old source still wins | hidden positive-minus-negative raw-`z` gap | no; only FunnyBird has a donor/source margin |
| Does species still organize the score? | source-species residual after exact source/donor values | species residual after exact concept and mask state | related and observational |
| Is the exact inserted value recognized? | controlled post-swap value confusion | no clean equivalent | unavailable in CUB |

### Model health comes before grounding

For exact concept `j`, the model predicts positive when `z_ij>0`. Balanced
accuracy gives positive and negative examples equal weight:

`balanced_accuracy = (positive recall + negative recall) / 2`.

If 70% of positive examples and 80% of negative examples are correct, balanced
accuracy is `(0.70+0.80)/2 = 0.75`; the aligned summary plots ordinary concept
error `1-0.75 = 0.25`. A large error says the output is difficult. It does not
say whether the error came from context, weak pixels, or noisy labels.

An output is **collapsed** when its raw score is effectively constant across all
images: `Q95(z)-Q05(z) <= 1e-8`. For example, returning `z=+2.1` for every image
always predicts “present.” Positive recall would misleadingly equal 1, negative
recall would equal 0, and balanced accuracy would equal 0.5. Such an output did
not learn a usable image distinction and cannot support a grounding claim.

The CUB70 model has two exactly collapsed outputs:
`has_throat_color::grey` is constant-positive and
`has_wing_pattern::multi-colored` is constant-negative. They remain visible as
negative health results and are excluded from positive grounding summaries.

### Direction of each CUB panel

| Panel type | A larger value means | Interpretation |
|---|---|---|
| **Data check: positive label / mask absent** | more positive labels lack a usable mapped mask | possible label/visibility conflict, but also possible missing annotation |
| **Health check: ordinary concept error** | worse positive/negative prediction | weak or difficult output; not automatically backwash |
| **Local evidence: visible - hidden raw `z`** | positive examples score higher when the region is visible | evidence that local pixels help; usually a good grounding sign |
| **Context evidence: hidden positive - negative raw `z`** | labels remain separated when the mapped mask is absent | context or unmeasured pixels remain informative |
| **Species context: residual spread** | species shift `z` after exact concept and mask state are centered | species-associated organization remains |

These quantities have different units and directions. They must not be added
into a synthetic “CUB backwash score.” Repeatedly unusual groups are stronger
observational candidates; only a controlled outcome can measure causal
backwash directly.


## 9b · Synthesis only: do the already measured contributors line up with the controlled part ordering?

**Question.** Synthesis only: do the already measured contributors line up with the controlled part ordering?

**Variables and prediction.** Place four separately defined part-level quantities in aligned panels: the controlled backwash-candidate rate, the same rate among swaps with at least 100 target pixels, the training label/mask conflict rate, and one minus exact donor-value recognition. Tail should be high across several contributor panels while wing and foot should be low if the proposed explanation matches the controlled outcome. The panels use different units and must not be added together.

**Method.** Use the same five-part order in every panel and print the exact table.

### Figure 9b · Synthesis only: do the already measured contributors line up with the controlled part ordering?

**How to read the figure.** All four panels use the same y-axis part order. Panel A is the fraction of
all swaps satisfying `response_delta>0 and m_cf<0`. Panel B repeats that
fraction only when the inserted target occupies at least 100 pixels. Panel C
is the fraction of original positive training labels removed by the matched
visibility rule. Panel D is one minus the post-swap inserted-value recognition
rate. Larger is worse in every panel, but the denominators and meanings differ,
so the bar heights must not be added. The shared ordering asks whether the
proposed contributors align with the controlled outcome.


In [ ]:
# ALT: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
if "PART_CONFLICT" not in globals():
    raise RuntimeError("Figure 9b requires the matched standard/RLv2 label records used in Figure 6b")
if "diag" not in globals():
    raise RuntimeError("Figure 9b requires the exact-value recognition results from Figure 7")
FB_SYN=pd.DataFrame(index=ORDER)
FB_SYN.index.name="part"
FB_SYN["controlled_backwash_rate"]=(S.groupby("part").responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["clear_visible_backwash_rate"]=(S[S.pixel_count_cf>=100].groupby("part")
                                          .responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["label_mask_conflict_rate"]=PART_CONFLICT.conflict_rate.reindex(ORDER)
FB_SYN["donor_value_error_rate"]=pd.Series({p:1-diag[p] for p in ORDER}).reindex(ORDER)
panels=[
    ("controlled_backwash_rate","A · OUTCOME: old source still wins"),
    ("clear_visible_backwash_rate","B · VISIBILITY CHECK: target ≥100 px"),
    ("label_mask_conflict_rate","C · DATA CHECK: label/mask conflict"),
    ("donor_value_error_rate","D · VISUAL DIFFICULTY: value misidentified"),
]
fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True)
for ax,(column,title) in zip(axes.flat,panels):
    ax.barh(np.arange(len(ORDER)),FB_SYN[column],color=[COLORS[p] for p in ORDER])
    ax.set_xlim(0,1); ax.set_title(title,fontsize=10); ax.set_xlabel("fraction")
    ax.set_yticks(np.arange(len(ORDER)),ORDER); ax.invert_yaxis()
    for y,value in enumerate(FB_SYN[column]):
        ax.text(value+.015,y,f"{value:.3f}",va="center",fontsize=8)
fig.suptitle("Figure 9b · Synthesis of earlier measurements in one part order\n(no new causal evidence)")
plt.tight_layout(); plt.show(); display(FB_SYN.round(3))


- **Method in one line:** We aligned four previously computed part-level summaries in one fixed order without fitting or adding them, so only their descriptive ordering—not percent explained—can be compared.


### Plain-language reference for Figure 9b

**Plain caption.** The observed part ordering lines up across controlled
backwash, clearly visible swaps, training label/mask conflict, and exact-
value error, but the four bars are not pieces that can be added.

**Terms and denominators.** Panel A uses all 1,000 swaps per part. Panel B
uses only swaps with at least 100 inserted-part pixels. Panel C divides
removed positive training labels by all original positives. Panel D is
one minus the inserted-value recognition rate. Every panel is a fraction,
but the populations and questions differ.

**Literal values.** Controlled event rates are tail `0.502`, beak
`0.200`, eye `0.089`, foot `0.032`, and wing `0.019`. With target area at
least 100 pixels they remain `0.372`, `0.131`, `0.052`, `0.017`, and
`0.010`. Label/mask conflict rates are `0.199`, `0.010`, `0.007`,
`0.001`, and less than `0.001`; donor-value error rates are `0.605`,
`0.220`, `0.100`, `0.035`, and `0.023` in the same part order.

**Interpretation.** The hardest controlled part is also the part with the
most invisible-positive supervision and wrong exact-value answers. That
agreement makes the proposed story plausible, but five correlated part
summaries cannot measure how much each cause contributed.

**Alternative.** Alignment can arise from correlated part properties,
and there are only five anatomical units.

**Discriminating test.** Use the same-source-image held-out prediction in Figure 9
and the matched RLv2 intervention for label-conflict causality.

**Verdict.** **KEEP**.

**Proof ledger.** Visibility-resistant events, label conflict, and exact-
value difficulty align descriptively with the controlled ordering. They
are not additive and do not fully explain the residual.

**Next question.** Does the concept-layer behavior have a meaningful
downstream species-prediction consequence?


## 10 · Is final concept margin associated with donor-species probability?

**Question.** Is final concept margin associated with donor-species probability?

**Variables and prediction.** Relate final concept margin to the model's donor-species probability, which is a different downstream quantity; this is an association, not an intervention on the margin. A small downstream change would limit the harm to explanation reliability rather than widespread class failure.

**Method.** Use independent final-margin bins and print bin counts.

### Figure 10 · Is final concept margin associated with donor-species probability?

**How to read the figure.** Swaps are divided into ten non-overlapping, approximately equal-count bins by final donor-minus-source concept
margin on the x-axis. The y-axis is the model's mean probability for the donor
species, with the number of rows printed per bin. This asks whether concept
grounding failure is associated with a downstream class quantity; it is intentionally the one
place where class probability, rather than raw concept `z`, is the outcome.


In [ ]:
# ALT: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
prob_col=next((c for c in ["p_cf_donor","p_donor_cf","donor_species_prob"] if c in S),None)
if prob_col is None:
    print("INCOMPLETE: swap CSV has no donor-species probability column")
else:
    D=S.copy(); D["margin_bin"]=pd.qcut(D.margin,10,duplicates="drop")
    Q=D.groupby("margin_bin",observed=True).agg(n=(prob_col,"size"),mean_margin=("margin","mean"),mean_donor_species_prob=(prob_col,"mean")).reset_index()
    fig,ax=plt.subplots(figsize=(7,4)); ax.plot(Q.mean_margin,Q.mean_donor_species_prob,"o-")
    for k,r in enumerate(Q.itertuples()):
        ax.annotate(f"n={r.n}",(r.mean_margin,r.mean_donor_species_prob),
                    fontsize=7,xytext=(3,8 if k%2==0 else -12),
                    textcoords="offset points")
    ax.axvline(0,color="black",lw=.8); ax.set_xlabel("mean final concept margin in bin")
    ax.set_ylabel("mean donor-species probability"); ax.set_title("Figure 10 · Association between concept margin and donor-species probability")
    plt.tight_layout(); plt.show(); display(Q.round(3))


- **Method in one line:** We divided all swaps into ten disjoint equal-count bins by final concept margin and averaged the frozen CBM's saved donor-species probability within each bin; no regression or new classifier was fitted.


    ### Plain-language reference for Figure 10

    **Plain caption.** A more donor-positive concept margin is associated with a modestly larger saved donor-species probability; this binned analysis does not intervene on the margin.

    **Terms and how to read it.** The x-axis is mean final concept margin inside one of ten disjoint bins. The y-axis is the saved model's mean donor-species probability. This is the only main figure using class probability rather than raw concept `z`.

    **Literal values.** Mean donor-species probability rises monotonically from approximately zero in negative-margin bins to 0.110 in the most donor-positive bin, with 499-501 rows per bin.

    **Interpretation.** When the donor concept finishes farther ahead, the model becomes more willing to predict the donor species. The probability still reaches only 11% in the strongest bin because the other four parts and the body still belong to the source bird.

**Strongest alternative explanation.** A one-part replacement need not make the whole donor species plausible because the unchanged body and other parts still belong to the source.

    **Discriminating test.** Replicate across seeds and compare class-logit changes, not only final probability.

    **Verdict.** **KEEP**.

    **Proof ledger.** `ACCEPTED FOR a monotone but modest single-swap downstream association; the primary harm here is explanation fidelity.`

    **Next question.** Does minimality change the accepted standard-CBM quantities?


## 11 · Standard-CBM evidence ledger

| Predicate or explanation | Direct measurement | Status after review |
|---|---|---|
| model outputs are usable | Figure 1 | `ACCEPTED FOR SEED-1 MODEL HEALTH` |
| interventions are valid | Figure 2 | `ACCEPTED FOR CONTROLLED ONE-PART REPLACEMENT` |
| controlled part replacement causes donorward movement | Figure 3 | `ACCEPTED AT PART LEVEL; POSITIVE-RESPONSE RATES 0.919-1.000, NOT EVERY ROW` |
| starting preference versus donor rise/source release | Figure 3b | `ACCEPTED ARITHMETIC DECOMPOSITION` |
| old source can remain stronger after that movement | Figure 4 | `ACCEPTED FOR GRADED CONTROLLED BACKWASH` |
| donor wins versus two distinct failure states | Figure 4b | `ACCEPTED OUTCOME PARTITION` |
| direction artifact excluded | Figure 5 | `ACCEPTED; ORDERING HOLDS BOTH DIRECTIONS` |
| visibility contribution | Figure 6 | `ACCEPTED AS CONTRIBUTOR, NOT SUFFICIENT` |
| training label/mask conflict measured | Figure 6b | `ACCEPTED DATA ASSOCIATION; CAUSAL TEST IS 02RL` |
| exact-value difficulty | Figure 7 | `ACCEPTED AS GRADED ASSOCIATION/CANDIDATE CONTRIBUTOR` |
| frequency/alternative-count explanation | Figure 7b | `MIXED; NO SUFFICIENT MONOTONE EXPLANATION` |
| source-species residual | Figure 8 | `DESCRIPTIVE ASSOCIATION ONLY` |
| species information beyond concept-label buckets | Figure 8b | `ACCEPTED FOR AVAILABILITY, NOT GROUNDING` |
| post-swap source retention distinguishes controlled failures | Figure 8c | `VALID DISCRIMINATING TEST, NO SUPPORT FOR THE PREDICTED TAIL-SPECIFIC CONTRAST` |
| progressively richer held-out grouping predictor | Figure 9 | `VISIBILITY IMPROVES HELD-OUT ERROR; EXACT VALUE/SPECIES DO NOT` |
| aligned contributor view | Figure 9b | `ACCEPTED DESCRIPTIVELY; NOT ADDITIVE OR CAUSAL` |
| downstream class association | Figure 10 | `ACCEPTED FOR MODEST MONOTONE ASSOCIATION; NOT A MARGIN INTERVENTION` |

### Limited conclusion

**Backwash exists in this seed-1 Standard CBM.** It is not necessary for
all parts to fail identically: the controlled predicate is row-level,
and its prevalence is graded from tail through wing. The renderer targets
one declared part while preserving the scene; 98.3% of cached replacement
RGB images visibly differ from their originals, while the remaining 1.7%
are retained for the visibility analysis. At the part-summary level,
positive donorward-response rates range from 91.9% to 100%; this is not a
claim that every individual swap responds. Within the measured subset that
responds but finishes with a negative margin, the final concept answer
nevertheless remains attached to the old source.

The proposed contributors and alternatives were investigated. Visibility
accounts for some held-out organization but leaves many clearly visible
tail events. Label/mask conflict and exact-value error closely match the
part ordering, with tail highest and wing/foot lowest, but their current
standard-model analyses are associations. Rarity/support is mixed.
Source species strongly appears in the learned concept representation
and in descriptive residuals, yet adding source species worsens held-out
margin prediction under the declared categorical estimator. Figure 8b
also supplies the crucial wing counterexample: species information is
abundant even where exact donor recognition and donorward movement are
strong enough that backwash is rare. Figure 8c then runs the narrower
outcome-specific test after exact source/donor values are accounted for.
Tail residual source decoding is lower in controlled failures than in donor
wins (`0.351` versus `0.400`), not higher as the proposed source-retention
mechanism predicted. Beak is higher (`0.340` versus `0.231`) and eye is
nearly unchanged (`0.135` versus `0.126`); wing and foot controlled-failure
groups lack the predeclared 25-original-image coverage for interpretation.
Thus source information remains available, but this test gives no support
for a universal or tail-specific source-retention explanation of the
controlled event.
Therefore the evidence does **not** support saying that backwash is fully
explained or that the measured contributors exhaust every causal pathway.

### Why the official MCBM `-3/+3` targets are not automatically the answer

This Standard CBM does not use an MCBM representation target. Notebook 03
asks that separate numerical question. The pinned MCBM source targets
`-3` for an absent concept and `+3` for a present concept. In an ideal
red-to-blue replacement, the donor coordinate would move `-3 -> +3`
(`+6`) while the removed source coordinate moves `+3 -> -3` (another
`+6` contribution to donor-minus-source margin), for total margin movement
`+12`. A hypothetical `-5/+5` target would analogously move the margin by
`+20`, not `+10`. But a representation penalty does not specify which
pixels must produce the target. A network can recognize species/body and
output the correct target on ordinary training images. Achieving the
desired counterfactual movement may still require swap-aware or spatial
grounding supervision; compression alone does not guarantee it.

### Explicit handoff through the decreasing-information chapters

| Requested follow-up | Where it belongs | What it must establish |
|---|---|---|
| Standard MCBM gamma sweep | notebook 03 | whether compression removes within-label species information and reduces the same fixed-swap event while health remains usable |
| relabelled CBM | notebook 02rl | causal effect of changing positive labels when the named part is invisible |
| relabelled MCBM | notebook 03rl | whether minimality and relabelling address distinct or overlapping routes |
| CUB70 Standard CBM/MCBM | notebooks 05/06 | whether FunnyBird-calibrated visibility, conflict, exact-value, support, and species signatures recur observationally in photographs |
| Full CUB | final 200-species stage | whether supported CUB70 signatures survive realistic scale and which matched questions lose adequate support |
| segmentation or spatial routing | later method decision | only pursue if the accepted evidence requires an explicit named-pixel path constraint |

The controlled FunnyBird swap remains the definition-quality base case
and calibration laboratory. CUB70 is the natural-image bridge: it can
repeat scientific questions using natural visibility, mask conflict,
exact-concept difficulty, support, and species-conditioned raw scores,
but those are weaker observational approximations—not donor/source
margins. Full CUB then asks whether the surviving signatures remain
detectable with 200 species and weaker matched support. As information
decreases, the claim must narrow rather than the metric being silently
weakened.

**Next report question.** Only after this ledger is reviewed may notebook
03 ask whether MCBM minimality changes the accepted standard-CBM
quantities. MCBM cannot replace this discovery.


# Methods appendix · measurements not used in the main claim

The reciprocal mask-deletion and randomized-patch experiments are retained
as method-development history. They did not reproduce the clean FunnyBird
control sufficiently to transfer their causal interpretation to CUB.

- reciprocal mask deletion: `METHOD NOT CALIBRATED FOR CROSS-DATASET CAUSAL COMPARISON`;
- randomized patch V1/V2: local pixel response was measurable in selected
  examples, but the all-part control was not calibrated and wing coverage was
  inadequate;
- none of these outcomes invalidates the validated renderer swap above.

Full artifacts and scripts remain under `analysis/paired_mask_deletion.py`,
`analysis/randomized_patch_masking.py`, and their output directories. They
are not rerun by this notebook.


# Provenance appendix

The table below records the live Git commit, input paths and SHA-256
hashes, row counts, seed, and the accepted fixed-render root. It is part
of the report: a stale HTML is not synchronized evidence.


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()
commit=subprocess.run(["git","rev-parse","HEAD"],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()
prov=[]
for role,path in [("fixed-render swap CSV",SWAP),("prediction export",PRED),("model checkpoint",MODEL)]:
    prov.append({"role":role,"path":str(path),"sha256":sha256_file(path)})
display(pd.DataFrame(prov)); display(pd.DataFrame([{"git_commit":commit,"seed":1,
    "swap_rows":len(S),"prediction_images":len(c_saved),"exact_concepts":len(CONCEPT_NAMES),
    "excluded_swap_rows":0,"accepted_render_root":str(SWAP.parent)}]))
